# <center>Deep Generative Models</center>
## <center>Seminar 11</center>

<center><b>April 16, 2026</b></center>

**Plan**:
1. Recap: From DDPM to Flow Matching (theory bridge)
2. SD 1.5 recap and baseline generation
3. SDXL: scaling the UNet paradigm
4. SD 3.5: MMDiT — from UNet to Transformers + Rectified Flow
5. FLUX: hybrid double/single stream transformer
6. FluxKontext: unified generation and in-context editing
7. NanoBanana: efficient MMDiT for production
8. Architecture evolution summary and trends

In [ ]:
!pip install -q diffusers transformers accelerate safetensors sentencepiece protobuf

import torch
import gc
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

assert torch.cuda.is_available(), "GPU required"
print(f"GPU: {torch.cuda.get_device_name(0)}")


## 1. From DDPM to Flow Matching — Theory Bridge

This section connects **Lectures 7–8** (DDPM, score matching) with **Lectures 11–12** (Flow Matching, Conditional FM). Understanding this transition is essential because it explains WHY modern diffusion models (SD 3.5, FLUX) abandoned noise prediction in favor of velocity prediction.

### 1.1 DDPM Recap

**Forward process** — gradually add Gaussian noise to data:

$$q(\mathbf{x}_t | \mathbf{x}_0) = \mathcal{N}\!\left(\sqrt{\bar\alpha_t}\,\mathbf{x}_0,\; (1-\bar\alpha_t)\,\mathbf{I}\right)$$

**Reparametrization trick** — sample $\mathbf{x}_t$ directly:

$$\mathbf{x}_t = \sqrt{\bar\alpha_t}\,\mathbf{x}_0 + \sqrt{1-\bar\alpha_t}\,\boldsymbol\epsilon, \quad \boldsymbol\epsilon \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$$

**Training objective** — predict the noise $\boldsymbol\epsilon$ that was added:

$$\mathcal{L}_{\text{DDPM}} = \mathbb{E}_{t,\, \mathbf{x}_0,\, \boldsymbol\epsilon}\left[\|\boldsymbol\epsilon - \boldsymbol\epsilon_\theta(\mathbf{x}_t, t)\|^2\right]$$

**Sampling** — iterative denoising with a learned noise schedule $\beta_t$:

$$\mathbf{x}_{t-1} = \frac{1}{\sqrt{\alpha_t}}\left(\mathbf{x}_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\boldsymbol\epsilon_\theta(\mathbf{x}_t, t)\right) + \sigma_t\,\mathbf{z}, \quad \mathbf{z} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$$

where $\alpha_t = 1 - \beta_t$ and $\bar\alpha_t = \prod_{s=1}^t \alpha_s$.

### 1.2 Problems with DDPM

Several limitations motivated the search for alternatives:

1. **Complex noise schedules**: The functions $\beta_t$, $\bar\alpha_t$ require careful tuning (linear, cosine, sigmoid schedules). Small changes significantly affect generation quality.

2. **Curved trajectories**: The variance-preserving forward process traces curved paths through data space. Approximating these curves requires many discretization steps — typically **50–1000 steps** for good quality.

3. **Stochastic sampling**: The standard DDPM sampler is an SDE. Deterministic sampling requires the DDIM trick (Song et al., 2020), which reformulates it as a probability flow ODE:
$$d\mathbf{x} = \left[\mathbf{f}(\mathbf{x}, t) - \frac{1}{2}g(t)^2 \nabla_{\mathbf{x}} \log p_t(\mathbf{x})\right] dt$$

4. **Theoretical complexity**: The full framework involves SDEs, score functions, and intricate connections between the forward and reverse processes (Lectures 7–10).

> **Key question**: Can we design a simpler generative process with straight trajectories that needs fewer steps?

### 1.3 Flow Matching — The Straight Path (Lecture 11)

**Core idea**: define a probability path $p_t(\mathbf{x})$ from noise $p_0 = \mathcal{N}(\mathbf{0}, \mathbf{I})$ to data $p_1 = p_{\text{data}}$, governed by an ODE:

$$\frac{d\mathbf{x}_t}{dt} = \mathbf{v}(\mathbf{x}_t, t), \quad t \in [0, 1]$$

The density $p_t$ must satisfy the **continuity equation** (Kolmogorov forward / Fokker–Planck):

$$\frac{\partial p_t(\mathbf{x})}{\partial t} = -\text{div}\!\left(\mathbf{v}(\mathbf{x}, t)\, p_t(\mathbf{x})\right)$$

**Flow Matching objective** — learn a velocity field $\mathbf{v}_\theta$ that matches the true velocity:

$$\mathcal{L}_{\text{FM}} = \mathbb{E}_{t \sim U[0,1]}\,\mathbb{E}_{\mathbf{x} \sim p_t(\mathbf{x})}\left[\|\mathbf{v}(\mathbf{x}, t) - \mathbf{v}_\theta(\mathbf{x}, t)\|^2\right]$$

**Problem**: both $p_t(\mathbf{x})$ and $\mathbf{v}(\mathbf{x}, t)$ are **intractable** — we don't know the marginal density path or the true velocity field.

> **Reference**: Lipman et al. "Flow Matching for Generative Modeling" (2022)

### 1.4 Conditional Flow Matching (Lecture 12)

**Solution**: condition on a data point $\mathbf{x}_1 \sim p_{\text{data}}$ and define a **conditional** probability path.

**Gaussian conditional path** (one-sided conditioning on endpoint $\mathbf{x}_1$):

$$p_t(\mathbf{x} | \mathbf{x}_1) = \mathcal{N}\!\left(\mathbf{x};\; t\,\mathbf{x}_1,\; (1-t)^2\,\mathbf{I}\right)$$

with boundaries: $p_0(\mathbf{x}|\mathbf{x}_1) = \mathcal{N}(\mathbf{0}, \mathbf{I})$ and $p_1(\mathbf{x}|\mathbf{x}_1) \to \delta(\mathbf{x} - \mathbf{x}_1)$.

**Conditional vector field** (derived from the Gaussian path):

$$\mathbf{v}_t(\mathbf{x} | \mathbf{x}_1) = \frac{\mathbf{x}_1 - \mathbf{x}}{1 - t}$$

**Conditional FM objective** (tractable!):

$$\mathcal{L}_{\text{CFM}} = \mathbb{E}_{t \sim U[0,1]}\,\mathbb{E}_{\mathbf{x}_1 \sim p_{\text{data}}}\,\mathbb{E}_{\mathbf{x} \sim p_t(\mathbf{x}|\mathbf{x}_1)}\left[\|\mathbf{v}_t(\mathbf{x} | \mathbf{x}_1) - \mathbf{v}_\theta(\mathbf{x}, t)\|^2\right]$$

> **Theorem** (Lecture 11): FM and CFM share the same optimal $\theta$ when $\text{supp}(p_t(\mathbf{x})) = \mathbb{R}^m$.

**Simplification with linear interpolation**: sample $\mathbf{x}_0 \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$, then:

$$\mathbf{x}_t = (1 - t)\,\mathbf{x}_0 + t\,\mathbf{x}_1$$

The target velocity becomes simply:

$$\mathbf{v}_t = \mathbf{x}_1 - \mathbf{x}_0$$

This gives **straight-line** paths from noise to data — each training sample defines a line segment from a noise point to a data point.

> **Reference**: Tong et al. "Improving and Generalizing Flow-Based Generative Models" (2023)

### 1.5 Rectified Flow

**Rectified Flow** = Flow Matching with straight paths + optional distillation for even fewer steps.

**Training procedure**:
1. Sample $t \sim U[0,1]$, $\;\mathbf{x}_0 \sim \mathcal{N}(\mathbf{0}, \mathbf{I})$, $\;\mathbf{x}_1 \sim p_{\text{data}}$
2. Construct interpolation: $\mathbf{x}_t = (1-t)\,\mathbf{x}_0 + t\,\mathbf{x}_1$
3. Minimize:

$$\mathcal{L}_{\text{RF}} = \mathbb{E}_{t,\,\mathbf{x}_0,\,\mathbf{x}_1}\left[\|\mathbf{v}_\theta(\mathbf{x}_t, t) - (\mathbf{x}_1 - \mathbf{x}_0)\|^2\right]$$

**Sampling** — simple Euler ODE integration from $t=0$ to $t=1$:

$$\mathbf{x}_{t+\Delta t} = \mathbf{x}_t + \Delta t \cdot \mathbf{v}_\theta(\mathbf{x}_t, t)$$

**Why it works**: straight paths are easier for the network to learn than curved DDPM trajectories. The Euler integrator introduces minimal error along straight lines, so **10–50 steps** suffice (vs 50–1000 for DDPM). With progressive distillation, this can be reduced to **1–4 steps**.

> **Reference**: Liu et al. "Flow Straight and Fast: Learning to Generate and Transfer Data with Rectified Flow" (2022)

### 1.6 DDPM vs Flow Matching — Comparison

| | **DDPM** | **Flow Matching / Rectified Flow** |
|---|---|---|
| **Training target** | Noise $\boldsymbol\epsilon$ | Velocity $\mathbf{v} = \mathbf{x}_1 - \mathbf{x}_0$ |
| **Forward process** | $\mathbf{x}_t = \sqrt{\bar\alpha_t}\,\mathbf{x}_0 + \sqrt{1-\bar\alpha_t}\,\boldsymbol\epsilon$ | $\mathbf{x}_t = (1-t)\,\mathbf{x}_0 + t\,\mathbf{x}_1$ |
| **Path geometry** | Curved (variance-preserving/exploding) | Straight lines |
| **Sampling** | SDE or DDIM ODE (complex) | Euler ODE (simple) |
| **Steps needed** | 50–1000 | 10–50 (4 with distillation) |
| **Schedule** | Complex $\beta_t$ with many design choices | Simple linear interpolation $t \in [0, 1]$ |
| **Theory** | Score matching + SDE (Lectures 7–10) | Continuity eq. + ODE (Lectures 11–12) |

The transition from DDPM to Rectified Flow is one of the **two paradigm shifts** that define the SD 1.5 → SD 3.5 evolution. The other is the architectural shift from UNet to Transformers, which we cover in Section 4.

---
*Next: let's revisit SD 1.5 as our baseline before exploring the newer architectures.*

### Curved vs Straight: Why it matters

<center>
<svg width="650" height="220" xmlns="http://www.w3.org/2000/svg" font-family="Arial, sans-serif" font-size="12">
  <text x="150" y="18" text-anchor="middle" font-size="14" font-weight="bold" fill="#ea4335">DDPM — curved paths</text>
  <line x1="30" y1="190" x2="280" y2="190" stroke="#ccc" stroke-width="1"/>
  <line x1="30" y1="190" x2="30" y2="35" stroke="#ccc" stroke-width="1"/>
  <text x="155" y="210" text-anchor="middle" fill="#999" font-size="10">t = 0 → T (1000 steps)</text>
  <path d="M 40,170 Q 80,60 130,80 Q 180,100 230,50" fill="none" stroke="#ea4335" stroke-width="2" opacity="0.7"/>
  <path d="M 40,150 Q 100,40 160,90 Q 200,120 250,45" fill="none" stroke="#ea4335" stroke-width="1.8" opacity="0.5"/>
  <path d="M 40,180 Q 90,80 140,110 Q 190,60 260,55" fill="none" stroke="#ea4335" stroke-width="1.8" opacity="0.5"/>
  <path d="M 40,160 Q 70,100 120,50 Q 170,80 240,40" fill="none" stroke="#ea4335" stroke-width="1.8" opacity="0.3"/>
  <circle cx="40" cy="170" r="4" fill="#ea4335"/>
  <text x="55" y="185" fill="#ea4335" font-size="10" font-weight="bold">noise</text>
  <circle cx="245" cy="47" r="4" fill="#ea4335"/>
  <text x="260" y="42" fill="#ea4335" font-size="10" font-weight="bold">data</text>
  <text x="150" y="145" text-anchor="middle" fill="#c5221f" font-size="10">50–1000 steps needed</text>
  <line x1="320" y1="20" x2="320" y2="200" stroke="#ddd" stroke-width="1.5" stroke-dasharray="5,3"/>
  <text x="490" y="18" text-anchor="middle" font-size="14" font-weight="bold" fill="#34a853">Rectified Flow — straight paths</text>
  <line x1="370" y1="190" x2="620" y2="190" stroke="#ccc" stroke-width="1"/>
  <line x1="370" y1="190" x2="370" y2="35" stroke="#ccc" stroke-width="1"/>
  <text x="495" y="210" text-anchor="middle" fill="#999" font-size="10">t = 0 → 1 (10–50 steps)</text>
  <line x1="380" y1="170" x2="580" y2="50" stroke="#34a853" stroke-width="2" opacity="0.7"/>
  <line x1="380" y1="150" x2="600" y2="45" stroke="#34a853" stroke-width="1.8" opacity="0.5"/>
  <line x1="380" y1="180" x2="590" y2="55" stroke="#34a853" stroke-width="1.8" opacity="0.5"/>
  <line x1="380" y1="160" x2="570" y2="40" stroke="#34a853" stroke-width="1.8" opacity="0.3"/>
  <circle cx="380" cy="170" r="4" fill="#34a853"/>
  <text x="395" y="185" fill="#34a853" font-size="10" font-weight="bold">noise</text>
  <circle cx="585" cy="50" r="4" fill="#34a853"/>
  <text x="600" y="44" fill="#34a853" font-size="10" font-weight="bold">data</text>
  <circle cx="430" cy="145" r="3" fill="#34a853" opacity="0.5"/>
  <circle cx="480" cy="115" r="3" fill="#34a853" opacity="0.5"/>
  <circle cx="530" cy="82" r="3" fill="#34a853" opacity="0.5"/>
  <text x="490" y="135" text-anchor="middle" fill="#1b5e20" font-size="10">10–50 steps enough!</text>
</svg>
</center>

Straight paths = less discretization error = fewer steps for the same quality.

---
*Next: let's revisit SD 1.5 as our baseline before exploring the newer architectures.*

## 2. SD 1.5 — The Baseline (Recap)

We built SD 1.5 from scratch in **Seminar 9**. Here is a quick recap of the architecture:

<center>
<svg width="700" height="220" xmlns="http://www.w3.org/2000/svg" font-family="Arial, sans-serif" font-size="13">
  <!-- Text Encoder -->
  <rect x="20" y="20" width="120" height="45" rx="8" fill="#e8f0fe" stroke="#4285f4" stroke-width="2"/>
  <text x="80" y="38" text-anchor="middle" font-weight="bold" fill="#1a73e8">Text</text>
  <text x="80" y="55" text-anchor="middle" fill="#555" font-size="11">"a photo of..."</text>
  <!-- Arrow -->
  <line x1="140" y1="42" x2="180" y2="42" stroke="#666" stroke-width="2" marker-end="url(#arrow)"/>
  <!-- CLIP -->
  <rect x="180" y="15" width="130" height="55" rx="8" fill="#fce8e6" stroke="#ea4335" stroke-width="2"/>
  <text x="245" y="38" text-anchor="middle" font-weight="bold" fill="#c5221f">CLIP ViT-L/14</text>
  <text x="245" y="55" text-anchor="middle" fill="#555" font-size="11">77 × 768</text>
  <!-- Arrow down to UNet -->
  <line x1="245" y1="70" x2="400" y2="105" stroke="#666" stroke-width="2" marker-end="url(#arrow)"/>
  <!-- Noise -->
  <rect x="20" y="110" width="120" height="45" rx="8" fill="#f3e8fd" stroke="#9334e6" stroke-width="2"/>
  <text x="80" y="128" text-anchor="middle" font-weight="bold" fill="#7627bb">Noise</text>
  <text x="80" y="145" text-anchor="middle" fill="#555" font-size="11">z ~ N(0, I)</text>
  <!-- Arrow -->
  <line x1="140" y1="132" x2="340" y2="132" stroke="#666" stroke-width="2" marker-end="url(#arrow)"/>
  <!-- UNet -->
  <rect x="340" y="90" width="180" height="80" rx="10" fill="#fef7e0" stroke="#f9ab00" stroke-width="2.5"/>
  <text x="430" y="118" text-anchor="middle" font-weight="bold" font-size="14" fill="#e37400">UNet (860M)</text>
  <text x="430" y="138" text-anchor="middle" fill="#555" font-size="11">ResBlock + CrossAttn</text>
  <text x="430" y="153" text-anchor="middle" fill="#555" font-size="11">+ SpatialTransformer</text>
  <!-- Arrow -->
  <line x1="520" y1="130" x2="560" y2="130" stroke="#666" stroke-width="2" marker-end="url(#arrow)"/>
  <!-- VAE -->
  <rect x="560" y="105" width="120" height="50" rx="8" fill="#e6f4ea" stroke="#34a853" stroke-width="2"/>
  <text x="620" y="125" text-anchor="middle" font-weight="bold" fill="#1e8e3e">VAE Decoder</text>
  <text x="620" y="142" text-anchor="middle" fill="#555" font-size="11">512 × 512</text>
  <!-- Arrow defs -->
  <defs><marker id="arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#666"/></marker></defs>
  <!-- Labels -->
  <text x="430" y="195" text-anchor="middle" fill="#888" font-size="12" font-style="italic">DDPM ε-prediction · 50 steps · 512×512</text>
</svg>
</center>

**Key specs**:
- **Denoiser**: UNet with ~860M parameters (ResBlocks, CrossAttention, SpatialTransformer blocks)
- **Text encoder**: CLIP ViT-L/14, producing 77 tokens × 768 dimensions
- **VAE**: Encoder/decoder for 8× spatial compression (512×512 → 64×64 latent, 4 channels)
- **Training**: DDPM with $\boldsymbol\epsilon$-prediction, trained on LAION-5B
- **Resolution**: 512×512

> **Reference**: Rombach et al. "High-Resolution Image Synthesis with Latent Diffusion Models" (2022)

### Classifier-Free Guidance & Negative Prompts

All SD-family models use **Classifier-Free Guidance (CFG)** to steer generation. This is also how **negative prompts** work.

During training, the model learns both conditional and unconditional generation (by randomly dropping the text condition). At inference, the two predictions are combined:

$$\tilde{\boldsymbol\epsilon}_\theta(\mathbf{x}_t, t, \mathbf{c}) = \boldsymbol\epsilon_\theta(\mathbf{x}_t, t, \varnothing) + w \cdot \bigl(\boldsymbol\epsilon_\theta(\mathbf{x}_t, t, \mathbf{c}) - \boldsymbol\epsilon_\theta(\mathbf{x}_t, t, \varnothing)\bigr)$$

where $w$ is the **guidance scale** (typically 7–9) and $\varnothing$ is the unconditional (empty) prompt.

**Negative prompt** replaces $\varnothing$ with a custom "anti-prompt" $\mathbf{c}_{\text{neg}}$:

$$\tilde{\boldsymbol\epsilon}_\theta = \boldsymbol\epsilon_\theta(\mathbf{x}_t, t, \mathbf{c}_{\text{neg}}) + w \cdot \bigl(\boldsymbol\epsilon_\theta(\mathbf{x}_t, t, \mathbf{c}_{\text{pos}}) - \boldsymbol\epsilon_\theta(\mathbf{x}_t, t, \mathbf{c}_{\text{neg}})\bigr)$$

The model is pushed **away** from $\mathbf{c}_{\text{neg}}$ and **toward** $\mathbf{c}_{\text{pos}}$.

<center>
<svg width="600" height="180" xmlns="http://www.w3.org/2000/svg" font-family="Arial, sans-serif" font-size="11">
  <defs><marker id="cfg" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#666"/></marker></defs>
  <!-- Noisy latent -->
  <rect x="10" y="60" width="90" height="40" rx="6" fill="#f3e8fd" stroke="#9334e6" stroke-width="1.5"/>
  <text x="55" y="78" text-anchor="middle" fill="#7627bb" font-size="10" font-weight="bold">Noisy x_t</text>
  <text x="55" y="92" text-anchor="middle" fill="#999" font-size="9">+ timestep t</text>
  <!-- Fork into two paths -->
  <line x1="100" y1="72" x2="140" y2="30" stroke="#666" stroke-width="1.5" marker-end="url(#cfg)"/>
  <line x1="100" y1="88" x2="140" y2="130" stroke="#666" stroke-width="1.5" marker-end="url(#cfg)"/>
  <!-- Conditional pass -->
  <rect x="140" y="10" width="150" height="40" rx="6" fill="#e6f4ea" stroke="#34a853" stroke-width="2"/>
  <text x="215" y="27" text-anchor="middle" fill="#1b5e20" font-size="10" font-weight="bold">UNet(x_t, c_pos)</text>
  <text x="215" y="42" text-anchor="middle" fill="#888" font-size="9">"astronaut on Mars"</text>
  <!-- Negative pass -->
  <rect x="140" y="110" width="150" height="40" rx="6" fill="#fce8e6" stroke="#ea4335" stroke-width="2"/>
  <text x="215" y="127" text-anchor="middle" fill="#c5221f" font-size="10" font-weight="bold">UNet(x_t, c_neg)</text>
  <text x="215" y="142" text-anchor="middle" fill="#888" font-size="9">"blurry, low quality"</text>
  <!-- Arrows to combine -->
  <line x1="290" y1="30" x2="340" y2="72" stroke="#34a853" stroke-width="1.5" marker-end="url(#cfg)"/>
  <line x1="290" y1="130" x2="340" y2="88" stroke="#ea4335" stroke-width="1.5" marker-end="url(#cfg)"/>
  <!-- Combine -->
  <rect x="340" y="55" width="140" height="50" rx="8" fill="#fef7e0" stroke="#f9ab00" stroke-width="2"/>
  <text x="410" y="74" text-anchor="middle" fill="#e37400" font-size="10" font-weight="bold">CFG Combine</text>
  <text x="410" y="92" text-anchor="middle" fill="#888" font-size="9">neg + w*(pos - neg)</text>
  <!-- Output -->
  <line x1="480" y1="80" x2="520" y2="80" stroke="#666" stroke-width="1.5" marker-end="url(#cfg)"/>
  <rect x="520" y="62" width="70" height="36" rx="6" fill="#ede7f6" stroke="#7c4dff" stroke-width="1.5"/>
  <text x="555" y="78" text-anchor="middle" fill="#4a148c" font-size="10" font-weight="bold">Guided</text>
  <text x="555" y="92" text-anchor="middle" fill="#4a148c" font-size="9">output</text>
  <!-- Labels -->
  <text x="160" y="78" text-anchor="middle" fill="#aaa" font-size="9">2 forward passes</text>
</svg>
</center>

**Example**: `negative_prompt="blurry, low quality, distorted"` pushes the model away from generating blurry/distorted outputs. The guidance scale $w$ controls how strongly.

In [ ]:
from diffusers import StableDiffusionPipeline

pipe_sd15 = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
    safety_checker=None,
).to("cuda")

print(f"UNet params: {sum(p.numel() for p in pipe_sd15.unet.parameters()) / 1e6:.0f}M")
print(f"Text encoder dim: {pipe_sd15.text_encoder.config.hidden_size}")

# Fixed prompt and seed — will reuse for all runnable models
PROMPT = "A photograph of an astronaut riding a horse on Mars, detailed, 8k"
NEGATIVE = "blurry, low quality, distorted"
SEED = 42

generator = torch.Generator("cuda").manual_seed(SEED)
img_sd15 = pipe_sd15(
    PROMPT, negative_prompt=NEGATIVE, generator=generator,
    num_inference_steps=50,
).images[0]

plt.figure(figsize=(6, 6))
plt.imshow(img_sd15)
plt.axis("off")
plt.title("SD 1.5 (512×512, 50 steps)")
plt.tight_layout()
plt.show()

In [ ]:
# Cleanup — free GPU memory before loading next model
del pipe_sd15
torch.cuda.empty_cache()
gc.collect()
print("SD 1.5 unloaded.")

## 3. SDXL — Scaling the UNet Paradigm

SDXL (2023) kept the UNet backbone but scaled it significantly. The key changes:

### 3.1 Dual Text Encoders

| Encoder | Output dim | Role |
|---------|-----------|------|
| CLIP ViT-L/14 | 768 | Visual-semantic alignment (same as SD 1.5) |
| OpenCLIP ViT-bigG/14 | 1280 | Richer language understanding |

The sequence embeddings are **concatenated** along the feature dimension: context = 768 + 1280 = **2048-dim**. This gives the UNet richer text conditioning.

Additionally, **pooled text embeddings** from both CLIP models are used as vector conditioning (similar to class conditioning in DiT) — concatenated and fed into the timestep MLP.

### 3.2 Larger UNet

~**3.5B parameters** (4× SD 1.5). More transformer blocks in the UNet, deeper cross-attention layers. Native resolution: **1024×1024**.

### 3.3 Micro-Conditioning

A key innovation: encode **target resolution** and **crop coordinates** as additional conditioning to solve the "everything looks like 256×256 upscaled" problem.

**Size conditioning** (original and target resolution):
$$\mathbf{c}_{\text{size}} = \text{FourierEmbed}(h_{\text{orig}}, w_{\text{orig}}, h_{\text{target}}, w_{\text{target}})$$

**Crop conditioning** (top-left crop position):
$$\mathbf{c}_{\text{crop}} = \text{FourierEmbed}(\text{top}, \text{left})$$

**Combined** — concatenated with the timestep embedding:
$$\mathbf{c} = [\mathbf{c}_{\text{time}};\; \mathbf{c}_{\text{size}};\; \mathbf{c}_{\text{crop}}] \;\;\text{fed to timestep MLP}$$

This allows the model to generate images at arbitrary aspect ratios and resolutions without quality degradation.

### 3.4 Two-Stage Refinement Pipeline

<center>
<svg width="600" height="100" xmlns="http://www.w3.org/2000/svg" font-family="Arial, sans-serif" font-size="12">
  <defs><marker id="a8" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#666"/></marker></defs>
  <rect x="5" y="30" width="80" height="36" rx="6" fill="#f3e8fd" stroke="#9334e6" stroke-width="1.5"/>
  <text x="45" y="53" text-anchor="middle" font-weight="bold" fill="#7627bb">Noise</text>
  <line x1="85" y1="48" x2="120" y2="48" stroke="#666" stroke-width="1.5" marker-end="url(#a8)"/>
  <rect x="120" y="22" width="180" height="52" rx="8" fill="#fef7e0" stroke="#f9ab00" stroke-width="2"/>
  <text x="210" y="42" text-anchor="middle" font-weight="bold" fill="#e37400">Base Model (3.5B)</text>
  <text x="210" y="60" text-anchor="middle" fill="#888" font-size="10">global structure, 1024²</text>
  <line x1="300" y1="48" x2="340" y2="48" stroke="#666" stroke-width="1.5" marker-end="url(#a8)"/>
  <rect x="340" y="22" width="185" height="52" rx="8" fill="#e8f0fe" stroke="#4285f4" stroke-width="2"/>
  <text x="432" y="42" text-anchor="middle" font-weight="bold" fill="#1a73e8">Refiner UNet</text>
  <text x="432" y="60" text-anchor="middle" fill="#888" font-size="10">fine details, last ~200 levels</text>
  <line x1="525" y1="48" x2="555" y2="48" stroke="#666" stroke-width="1.5" marker-end="url(#a8)"/>
  <rect x="555" y="30" width="40" height="36" rx="4" fill="#e6f4ea" stroke="#34a853" stroke-width="1.5"/>
  <text x="575" y="53" text-anchor="middle" fill="#1b5e20" font-size="11">IMG</text>
</svg>
</center>

The **base** model handles composition and structure; the **refiner** operates as an expert denoiser on the final noise levels, improving fine detail and texture quality.

### 3.5 SD 1.5 vs SDXL — Architecture Comparison

<center>
<svg width="660" height="310" xmlns="http://www.w3.org/2000/svg" font-family="Arial, sans-serif" font-size="12">
  <defs><marker id="a9" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#666"/></marker></defs>
  <text x="120" y="18" text-anchor="middle" font-size="15" font-weight="bold" fill="#444">SD 1.5</text>
  <rect x="45" y="32" width="150" height="36" rx="7" fill="#fce8e6" stroke="#ea4335" stroke-width="2"/>
  <text x="120" y="47" text-anchor="middle" font-weight="bold" fill="#c5221f" font-size="11">CLIP ViT-L</text>
  <text x="120" y="62" text-anchor="middle" fill="#888" font-size="10">768-dim</text>
  <line x1="120" y1="68" x2="120" y2="95" stroke="#666" stroke-width="1.5" marker-end="url(#a9)"/>
  <rect x="35" y="95" width="170" height="70" rx="9" fill="#fef7e0" stroke="#f9ab00" stroke-width="2.5"/>
  <text x="120" y="120" text-anchor="middle" font-weight="bold" font-size="14" fill="#e37400">UNet</text>
  <text x="120" y="137" text-anchor="middle" fill="#777" font-size="11">860M · 512²</text>
  <text x="120" y="155" text-anchor="middle" fill="#777" font-size="10">CrossAttn only</text>
  <text x="120" y="190" text-anchor="middle" fill="#aaa" font-size="10" font-style="italic">DDPM ε-prediction</text>
  <line x1="265" y1="8" x2="265" y2="295" stroke="#ddd" stroke-width="1.5" stroke-dasharray="6,4"/>
  <text x="462" y="18" text-anchor="middle" font-size="15" font-weight="bold" fill="#444">SDXL</text>
  <rect x="310" y="32" width="120" height="36" rx="7" fill="#fce8e6" stroke="#ea4335" stroke-width="2"/>
  <text x="370" y="47" text-anchor="middle" font-weight="bold" fill="#c5221f" font-size="11">CLIP ViT-L</text>
  <text x="370" y="62" text-anchor="middle" fill="#888" font-size="10">768</text>
  <rect x="450" y="32" width="150" height="36" rx="7" fill="#fce8e6" stroke="#ea4335" stroke-width="2"/>
  <text x="525" y="47" text-anchor="middle" font-weight="bold" fill="#c5221f" font-size="11">OpenCLIP bigG</text>
  <text x="525" y="62" text-anchor="middle" fill="#888" font-size="10">1280</text>
  <line x1="370" y1="68" x2="420" y2="82" stroke="#666" stroke-width="1.5" marker-end="url(#a9)"/>
  <line x1="525" y1="68" x2="476" y2="82" stroke="#666" stroke-width="1.5" marker-end="url(#a9)"/>
  <rect x="388" y="80" width="120" height="24" rx="5" fill="#ede7f6" stroke="#7c4dff" stroke-width="1.5"/>
  <text x="448" y="97" text-anchor="middle" fill="#4a148c" font-size="10" font-weight="bold">Concat → 2048</text>
  <line x1="448" y1="104" x2="448" y2="126" stroke="#666" stroke-width="1.5" marker-end="url(#a9)"/>
  <rect x="350" y="126" width="196" height="70" rx="9" fill="#fef7e0" stroke="#f9ab00" stroke-width="2.5"/>
  <text x="448" y="151" text-anchor="middle" font-weight="bold" font-size="14" fill="#e37400">UNet</text>
  <text x="448" y="168" text-anchor="middle" fill="#777" font-size="11">3.5B · 1024²</text>
  <text x="448" y="186" text-anchor="middle" fill="#777" font-size="10">CrossAttn + micro-cond</text>
  <line x1="448" y1="196" x2="448" y2="216" stroke="#666" stroke-width="1.5" marker-end="url(#a9)"/>
  <rect x="370" y="216" width="156" height="28" rx="6" fill="#e8f0fe" stroke="#4285f4" stroke-width="1.5"/>
  <text x="448" y="235" text-anchor="middle" fill="#1a73e8" font-size="11" font-weight="bold">+ Refiner UNet</text>
  <rect x="370" y="252" width="156" height="24" rx="5" fill="#e6f4ea" stroke="#34a853" stroke-width="1.5"/>
  <text x="448" y="269" text-anchor="middle" fill="#1b5e20" font-size="10" font-weight="bold">+ Micro-conditioning</text>
  <text x="448" y="295" text-anchor="middle" fill="#aaa" font-size="10" font-style="italic">Still DDPM ε-prediction</text>
</svg>
</center>

**Still DDPM-based** — same $\boldsymbol\epsilon$-prediction training, same noise schedules. The improvements come entirely from scaling and engineering.

> **Reference**: Podell et al. "SDXL: Improving Latent Diffusion Models for High-Resolution Image Synthesis" (2023)

---
*Next: the big paradigm shift — SD 3.5 replaces both the architecture AND the training objective.*

## 4. SD 3.5 — The Transformer Revolution (MMDiT + Rectified Flow)

This is the **biggest paradigm shift** in the Stable Diffusion family. Two fundamental changes happen simultaneously:

1. **Architecture**: UNet → **MMDiT** (Multimodal Diffusion Transformer)
2. **Training**: DDPM $\boldsymbol\epsilon$-prediction → **Rectified Flow** $\mathbf{v}$-prediction (covered in Section 1)

### SD 3.5 Full Pipeline

<center>
<svg width="660" height="180" xmlns="http://www.w3.org/2000/svg" font-family="Arial, sans-serif" font-size="11">
  <defs><marker id="tp" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#666"/></marker></defs>
  <rect x="5" y="10" width="95" height="55" rx="6" fill="#fce8e6" stroke="#ea4335" stroke-width="1.5"/>
  <text x="52" y="28" text-anchor="middle" font-weight="bold" fill="#c5221f" font-size="10">CLIP ViT-L</text>
  <text x="52" y="42" text-anchor="middle" fill="#888" font-size="9">768-dim</text>
  <text x="52" y="56" text-anchor="middle" fill="#888" font-size="9">pooled + seq</text>
  <rect x="5" y="72" width="95" height="44" rx="6" fill="#fce8e6" stroke="#ea4335" stroke-width="1.5"/>
  <text x="52" y="88" text-anchor="middle" font-weight="bold" fill="#c5221f" font-size="10">OpenCLIP bigG</text>
  <text x="52" y="104" text-anchor="middle" fill="#888" font-size="9">1280-dim</text>
  <rect x="5" y="124" width="95" height="44" rx="6" fill="#fce8e6" stroke="#ea4335" stroke-width="1.5"/>
  <text x="52" y="140" text-anchor="middle" font-weight="bold" fill="#c5221f" font-size="10">T5-XXL</text>
  <text x="52" y="156" text-anchor="middle" fill="#888" font-size="9">4096-dim, 4.7B</text>
  <line x1="100" y1="37" x2="130" y2="80" stroke="#666" stroke-width="1.2" marker-end="url(#tp)"/>
  <line x1="100" y1="94" x2="130" y2="88" stroke="#666" stroke-width="1.2" marker-end="url(#tp)"/>
  <line x1="100" y1="146" x2="130" y2="96" stroke="#666" stroke-width="1.2" marker-end="url(#tp)"/>
  <rect x="130" y="68" width="120" height="40" rx="6" fill="#ede7f6" stroke="#7c4dff" stroke-width="1.5"/>
  <text x="190" y="84" text-anchor="middle" fill="#4a148c" font-size="10" font-weight="bold">Project + Pad</text>
  <text x="190" y="98" text-anchor="middle" fill="#4a148c" font-size="9">→ 4096-dim joint</text>
  <line x1="250" y1="88" x2="285" y2="88" stroke="#666" stroke-width="1.5" marker-end="url(#tp)"/>
  <rect x="285" y="5" width="50" height="30" rx="5" fill="#f3e8fd" stroke="#9334e6" stroke-width="1.5"/>
  <text x="310" y="18" text-anchor="middle" fill="#7627bb" font-size="9" font-weight="bold">Noise</text>
  <text x="310" y="30" text-anchor="middle" fill="#999" font-size="8">latent</text>
  <line x1="310" y1="35" x2="310" y2="55" stroke="#666" stroke-width="1.2" marker-end="url(#tp)"/>
  <rect x="268" y="55" width="200" height="70" rx="10" fill="#fef7e0" stroke="#f9ab00" stroke-width="2.5"/>
  <text x="368" y="78" text-anchor="middle" font-weight="bold" font-size="13" fill="#e37400">MMDiT (2.5B)</text>
  <text x="368" y="95" text-anchor="middle" fill="#888" font-size="10">24 layers, joint self-attn</text>
  <text x="368" y="110" text-anchor="middle" fill="#888" font-size="9">Rectified Flow v-prediction</text>
  <rect x="285" y="140" width="85" height="25" rx="4" fill="#fff3e0" stroke="#ff9800" stroke-width="1"/>
  <text x="327" y="157" text-anchor="middle" fill="#e65100" font-size="9">pooled → adaLN</text>
  <line x1="190" y1="108" x2="285" y2="148" stroke="#ff9800" stroke-width="1" stroke-dasharray="3,2"/>
  <line x1="468" y1="90" x2="505" y2="90" stroke="#666" stroke-width="1.5" marker-end="url(#tp)"/>
  <rect x="505" y="68" width="100" height="42" rx="7" fill="#e6f4ea" stroke="#34a853" stroke-width="2"/>
  <text x="555" y="86" text-anchor="middle" font-weight="bold" fill="#1b5e20">VAE Decode</text>
  <text x="555" y="100" text-anchor="middle" fill="#888" font-size="9">16ch → 1024²</text>
  <line x1="605" y1="89" x2="635" y2="89" stroke="#666" stroke-width="1.5" marker-end="url(#tp)"/>
  <text x="648" y="93" fill="#333" font-size="13">IMG</text>
</svg>
</center>

### 4.1 DiT — Diffusion Transformer (Peebles & Xie, 2023)

Before MMDiT, let's understand **DiT** — the stepping stone from UNet to transformers.

**Key idea**: replace the UNet with a Vision Transformer (ViT).

**Patchification** — convert image to a sequence of tokens (just like ViT):

$$\mathbf{x} \in \mathbb{R}^{H \times W \times C} \;\;\xrightarrow{\text{patch embed}}\;\; \mathbf{z} \in \mathbb{R}^{N \times D}, \quad N = \frac{HW}{p^2}$$

where $p$ is the patch size (typically 2 for latent diffusion).

**Conditioning via adaLN-Zero** — adaptive layer norm replaces ResBlock time embedding:

$$\text{adaLN}(\mathbf{h}, \mathbf{c}) = \gamma(\mathbf{c}) \cdot \text{LayerNorm}(\mathbf{h}) + \beta(\mathbf{c})$$

where $\gamma, \beta$ are predicted from conditioning $\mathbf{c} = \text{MLP}(t_{\text{emb}} + \text{class}_{\text{emb}})$. The "Zero" means gate parameters $\alpha$ are **initialized to zero**, making each block initially an identity function — this stabilizes training.

**Text conditioning**: via **cross-attention** only:

$$\text{CrossAttn}(Q_{\text{img}}, K_{\text{text}}, V_{\text{text}})$$

<center>
<svg width="480" height="420" xmlns="http://www.w3.org/2000/svg" font-family="Arial, sans-serif" font-size="12">
  <defs><marker id="arr7" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#666"/></marker></defs>
  <rect x="1" y="1" width="478" height="418" rx="12" fill="none" stroke="#ccc" stroke-width="1.5" stroke-dasharray="6,3"/>
  <text x="200" y="25" text-anchor="middle" font-size="15" font-weight="bold" fill="#333">DiT Block</text>
  <!-- Input -->
  <rect x="100" y="42" width="200" height="32" rx="6" fill="#e8f0fe" stroke="#4285f4" stroke-width="2"/>
  <text x="200" y="63" text-anchor="middle" font-weight="bold" fill="#1a73e8">Image Tokens</text>
  <line x1="200" y1="74" x2="200" y2="95" stroke="#666" stroke-width="1.5" marker-end="url(#arr7)"/>
  <!-- adaLN 1 -->
  <rect x="90" y="95" width="220" height="32" rx="6" fill="#fff3e0" stroke="#ff9800" stroke-width="1.5"/>
  <text x="200" y="116" text-anchor="middle" fill="#e65100" font-size="11">adaLN-Zero (timestep + class)</text>
  <!-- conditioning arrow -->
  <rect x="350" y="88" width="110" height="46" rx="6" fill="#f3e8fd" stroke="#9334e6" stroke-width="1.5"/>
  <text x="405" y="106" text-anchor="middle" fill="#7627bb" font-size="10" font-weight="bold">Conditioning</text>
  <text x="405" y="122" text-anchor="middle" fill="#999" font-size="9">t<tspan font-size="8" baseline-shift="sub">emb</tspan> + class<tspan font-size="8" baseline-shift="sub">emb</tspan></text>
  <line x1="350" y1="111" x2="310" y2="111" stroke="#9334e6" stroke-width="1.5" marker-end="url(#arr7)"/>
  <line x1="200" y1="127" x2="200" y2="152" stroke="#666" stroke-width="1.5" marker-end="url(#arr7)"/>
  <!-- Self-Attention -->
  <rect x="75" y="152" width="250" height="38" rx="7" fill="#e8f5e9" stroke="#34a853" stroke-width="2"/>
  <text x="200" y="170" text-anchor="middle" font-weight="bold" fill="#1b5e20">Self-Attention</text>
  <text x="200" y="184" text-anchor="middle" fill="#555" font-size="10">(image tokens only)</text>
  <line x1="200" y1="190" x2="200" y2="215" stroke="#666" stroke-width="1.5" marker-end="url(#arr7)"/>
  <!-- Cross-Attention -->
  <rect x="75" y="215" width="250" height="38" rx="7" fill="#fce4ec" stroke="#e91e63" stroke-width="2"/>
  <text x="200" y="233" text-anchor="middle" font-weight="bold" fill="#880e4f">Cross-Attention</text>
  <text x="200" y="247" text-anchor="middle" fill="#555" font-size="10">Q = image, K/V = text</text>
  <!-- Text encoder arrow -->
  <rect x="355" y="208" width="105" height="52" rx="6" fill="#fce8e6" stroke="#ea4335" stroke-width="1.5"/>
  <text x="407" y="228" text-anchor="middle" fill="#c5221f" font-size="10" font-weight="bold">Text Encoder</text>
  <text x="407" y="244" text-anchor="middle" fill="#999" font-size="9">K<tspan font-size="8" baseline-shift="sub">text</tspan>, V<tspan font-size="8" baseline-shift="sub">text</tspan></text>
  <line x1="355" y1="234" x2="325" y2="234" stroke="#ea4335" stroke-width="1.5" marker-end="url(#arr7)"/>
  <!-- one-way label -->
  <text x="340" y="268" fill="#ea4335" font-size="10" font-weight="bold">Image ← Text</text>
  <text x="340" y="280" fill="#888" font-size="9">(one-way only!)</text>
  <line x1="200" y1="253" x2="200" y2="290" stroke="#666" stroke-width="1.5" marker-end="url(#arr7)"/>
  <!-- adaLN 2 -->
  <rect x="90" y="290" width="220" height="32" rx="6" fill="#fff3e0" stroke="#ff9800" stroke-width="1.5"/>
  <text x="200" y="311" text-anchor="middle" fill="#e65100" font-size="11">adaLN-Zero</text>
  <line x1="200" y1="322" x2="200" y2="348" stroke="#666" stroke-width="1.5" marker-end="url(#arr7)"/>
  <!-- FFN -->
  <rect x="110" y="348" width="180" height="35" rx="6" fill="#e8f0fe" stroke="#4285f4" stroke-width="2"/>
  <text x="200" y="370" text-anchor="middle" font-weight="bold" fill="#1a73e8">FFN (MLP)</text>
  <line x1="200" y1="383" x2="200" y2="408" stroke="#666" stroke-width="1.5" marker-end="url(#arr7)"/>
  <!-- Output -->
  <text x="200" y="418" text-anchor="middle" fill="#1a73e8" font-weight="bold" font-size="11">output tokens</text>
</svg>
</center>

**Scaling**: DiT showed that transformer architectures scale better than UNet — larger models consistently improve FID scores.

> **Reference**: Peebles & Xie "Scalable Diffusion Models with Transformers" (2023)

### 4.2 MMDiT — Multimodal Diffusion Transformer (Esser et al., 2024)

**Problem with DiT**: text tokens interact with image tokens only via cross-attention. Information flows **one direction** — image queries attend to text keys/values, but text never "sees" the image.

**MMDiT solution**: concatenate image and text tokens, run **joint self-attention**.

**Step 1 — Separate Q/K/V projections per modality**:

$$\mathbf{Q}_{\text{img}} = \mathbf{z}_{\text{img}}\,\mathbf{W}_Q^{\text{img}}, \quad \mathbf{Q}_{\text{txt}} = \mathbf{z}_{\text{txt}}\,\mathbf{W}_Q^{\text{txt}}$$

$$\mathbf{K}_{\text{img}} = \mathbf{z}_{\text{img}}\,\mathbf{W}_K^{\text{img}}, \quad \mathbf{K}_{\text{txt}} = \mathbf{z}_{\text{txt}}\,\mathbf{W}_K^{\text{txt}}$$

$$\mathbf{V}_{\text{img}} = \mathbf{z}_{\text{img}}\,\mathbf{W}_V^{\text{img}}, \quad \mathbf{V}_{\text{txt}} = \mathbf{z}_{\text{txt}}\,\mathbf{W}_V^{\text{txt}}$$

**Step 2 — Concatenate and apply joint attention**:

$$\mathbf{Q} = [\mathbf{Q}_{\text{img}};\; \mathbf{Q}_{\text{txt}}], \quad \mathbf{K} = [\mathbf{K}_{\text{img}};\; \mathbf{K}_{\text{txt}}], \quad \mathbf{V} = [\mathbf{V}_{\text{img}};\; \mathbf{V}_{\text{txt}}]$$

$$\text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\!\left(\frac{\mathbf{Q}\mathbf{K}^\top}{\sqrt{d}}\right)\mathbf{V}$$

**Step 3 — Split back and apply separate FFNs**:

$$\mathbf{z}_{\text{img}}' = \text{FFN}_{\text{img}}\!\left(\text{JointAttn}(\mathbf{z})[:N_{\text{img}}]\right)$$

$$\mathbf{z}_{\text{txt}}' = \text{FFN}_{\text{txt}}\!\left(\text{JointAttn}(\mathbf{z})[N_{\text{img}}:]\right)$$

### MMDiT Block Diagram

<center>
<svg width="620" height="420" xmlns="http://www.w3.org/2000/svg" font-family="Arial, sans-serif" font-size="12">
  <defs>
    <marker id="arr2" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#666"/></marker>
  </defs>
  <!-- Title -->
  <rect x="1" y="1" width="618" height="418" rx="12" fill="none" stroke="#ccc" stroke-width="1.5" stroke-dasharray="6,3"/>
  <text x="310" y="25" text-anchor="middle" font-size="15" font-weight="bold" fill="#333">MMDiT Block</text>
  <!-- Image stream (left) -->
  <rect x="40" y="50" width="140" height="40" rx="6" fill="#e8f0fe" stroke="#4285f4" stroke-width="2"/>
  <text x="110" y="75" text-anchor="middle" font-weight="bold" fill="#1a73e8">Image Tokens</text>
  <line x1="110" y1="90" x2="110" y2="120" stroke="#666" stroke-width="1.5" marker-end="url(#arr2)"/>
  <!-- adaLN img -->
  <rect x="55" y="120" width="110" height="32" rx="5" fill="#fff3e0" stroke="#ff9800" stroke-width="1.5"/>
  <text x="110" y="141" text-anchor="middle" fill="#e65100" font-size="11">adaLN<tspan font-size="9" baseline-shift="sub">img</tspan></text>
  <line x1="110" y1="152" x2="110" y2="175" stroke="#666" stroke-width="1.5" marker-end="url(#arr2)"/>
  <!-- Q K V img -->
  <rect x="30" y="175" width="160" height="35" rx="5" fill="#fce4ec" stroke="#e91e63" stroke-width="1.5"/>
  <text x="110" y="197" text-anchor="middle" fill="#880e4f" font-size="11">Q<tspan font-size="9" baseline-shift="sub">img</tspan>, K<tspan font-size="9" baseline-shift="sub">img</tspan>, V<tspan font-size="9" baseline-shift="sub">img</tspan></text>
  <!-- Text stream (right) -->
  <rect x="440" y="50" width="140" height="40" rx="6" fill="#fce8e6" stroke="#ea4335" stroke-width="2"/>
  <text x="510" y="75" text-anchor="middle" font-weight="bold" fill="#c5221f">Text Tokens</text>
  <line x1="510" y1="90" x2="510" y2="120" stroke="#666" stroke-width="1.5" marker-end="url(#arr2)"/>
  <!-- adaLN txt -->
  <rect x="455" y="120" width="110" height="32" rx="5" fill="#fff3e0" stroke="#ff9800" stroke-width="1.5"/>
  <text x="510" y="141" text-anchor="middle" fill="#e65100" font-size="11">adaLN<tspan font-size="9" baseline-shift="sub">txt</tspan></text>
  <line x1="510" y1="152" x2="510" y2="175" stroke="#666" stroke-width="1.5" marker-end="url(#arr2)"/>
  <!-- Q K V txt -->
  <rect x="430" y="175" width="160" height="35" rx="5" fill="#fce4ec" stroke="#e91e63" stroke-width="1.5"/>
  <text x="510" y="197" text-anchor="middle" fill="#880e4f" font-size="11">Q<tspan font-size="9" baseline-shift="sub">txt</tspan>, K<tspan font-size="9" baseline-shift="sub">txt</tspan>, V<tspan font-size="9" baseline-shift="sub">txt</tspan></text>
  <!-- Arrows into joint attention -->
  <line x1="110" y1="210" x2="220" y2="245" stroke="#4285f4" stroke-width="2" marker-end="url(#arr2)"/>
  <line x1="510" y1="210" x2="400" y2="245" stroke="#ea4335" stroke-width="2" marker-end="url(#arr2)"/>
  <!-- Joint Self-Attention (central) -->
  <rect x="190" y="240" width="240" height="48" rx="8" fill="#e8f5e9" stroke="#34a853" stroke-width="2.5"/>
  <text x="310" y="260" text-anchor="middle" font-weight="bold" font-size="13" fill="#1b5e20">Joint Self-Attention</text>
  <text x="310" y="278" text-anchor="middle" fill="#555" font-size="10">(all tokens attend to all tokens)</text>
  <!-- Arrows out -->
  <line x1="245" y1="288" x2="110" y2="320" stroke="#4285f4" stroke-width="2" marker-end="url(#arr2)"/>
  <line x1="375" y1="288" x2="510" y2="320" stroke="#ea4335" stroke-width="2" marker-end="url(#arr2)"/>
  <!-- FFN img -->
  <rect x="50" y="320" width="120" height="35" rx="6" fill="#e8f0fe" stroke="#4285f4" stroke-width="2"/>
  <text x="110" y="342" text-anchor="middle" font-weight="bold" fill="#1a73e8" font-size="12">FFN<tspan font-size="9" baseline-shift="sub">img</tspan></text>
  <!-- FFN txt -->
  <rect x="450" y="320" width="120" height="35" rx="6" fill="#fce8e6" stroke="#ea4335" stroke-width="2"/>
  <text x="510" y="342" text-anchor="middle" font-weight="bold" fill="#c5221f" font-size="12">FFN<tspan font-size="9" baseline-shift="sub">txt</tspan></text>
  <!-- Output arrows -->
  <line x1="110" y1="355" x2="110" y2="390" stroke="#666" stroke-width="1.5" marker-end="url(#arr2)"/>
  <line x1="510" y1="355" x2="510" y2="390" stroke="#666" stroke-width="1.5" marker-end="url(#arr2)"/>
  <!-- Output labels -->
  <text x="110" y="408" text-anchor="middle" fill="#1a73e8" font-weight="bold" font-size="11">img tokens'</text>
  <text x="510" y="408" text-anchor="middle" fill="#c5221f" font-weight="bold" font-size="11">txt tokens'</text>
  <!-- Bidirectional label -->
  <text x="310" y="312" text-anchor="middle" fill="#34a853" font-size="11" font-weight="bold">Image ↔ Text</text>
</svg>
</center>

**Key insight**: now text **sees** the image and image **sees** the text — **bidirectional** information flow. This dramatically improves text-image alignment, especially for complex prompts.

### DiT vs MMDiT

| | **DiT** | **MMDiT** |
|---|---|---|
| Text injection | Cross-attention | Joint self-attention |
| Information flow | Image ← Text (unidirectional) | Image ↔ Text (bidirectional) |
| Q, K, V weights | Shared across modalities | Separate per modality |
| FFN | Shared | Separate per modality |
| Text understanding | Good | Better (text sees image context) |

### 4.3 Triple Text Encoder in SD 3.5

SD 3.5 uses **three** text encoders for maximum text understanding:

| Encoder | Params | Output dim | Provides | Role |
|---------|--------|-----------|----------|------|
| CLIP ViT-L/14 | 123M | 768 | Sequence + pooled | Visual-semantic alignment |
| OpenCLIP ViT-bigG/14 | 694M | 1280 | Sequence + pooled | Richer language |
| T5-XXL | 4.7B | 4096 | Sequence only | Long text, typography, complex prompts |

**How embeddings are combined**:
- Sequence embeddings from all three encoders are **projected/padded to 4096 dimensions** and concatenated along the token axis → joint context tensor of shape `[batch, ~154, 4096]`
- **Pooled embeddings** from the two CLIP models are concatenated and added to the timestep embedding → adaLN conditioning

**T5 can be dropped at inference** (`text_encoder_3=None`) to save ~10 GB VRAM. Quality degrades mainly on complex/long prompts and text rendering.

### How the Triple Encoder Feeds MMDiT

<center>
<svg width="580" height="200" xmlns="http://www.w3.org/2000/svg" font-family="Arial, sans-serif" font-size="11">
  <defs><marker id="te" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#666"/></marker></defs>
  <text x="290" y="15" text-anchor="middle" font-size="13" font-weight="bold" fill="#333">Text Encoding in SD 3.5</text>
  <rect x="10" y="30" width="120" height="48" rx="6" fill="#fce8e6" stroke="#ea4335" stroke-width="1.5"/>
  <text x="70" y="48" text-anchor="middle" font-weight="bold" fill="#c5221f" font-size="10">CLIP ViT-L</text>
  <text x="70" y="64" text-anchor="middle" fill="#888" font-size="9">77 tok × 768</text>
  <rect x="10" y="88" width="120" height="48" rx="6" fill="#fce8e6" stroke="#ea4335" stroke-width="1.5"/>
  <text x="70" y="106" text-anchor="middle" font-weight="bold" fill="#c5221f" font-size="10">OpenCLIP bigG</text>
  <text x="70" y="122" text-anchor="middle" fill="#888" font-size="9">77 tok × 1280</text>
  <rect x="10" y="146" width="120" height="48" rx="6" fill="#fce8e6" stroke="#ea4335" stroke-width="1.5"/>
  <text x="70" y="164" text-anchor="middle" font-weight="bold" fill="#c5221f" font-size="10">T5-XXL (4.7B)</text>
  <text x="70" y="180" text-anchor="middle" fill="#888" font-size="9">77 tok × 4096</text>
  <line x1="130" y1="54" x2="175" y2="100" stroke="#666" stroke-width="1.2" marker-end="url(#te)"/>
  <line x1="130" y1="112" x2="175" y2="107" stroke="#666" stroke-width="1.2" marker-end="url(#te)"/>
  <line x1="130" y1="170" x2="175" y2="115" stroke="#666" stroke-width="1.2" marker-end="url(#te)"/>
  <rect x="175" y="85" width="130" height="45" rx="6" fill="#ede7f6" stroke="#7c4dff" stroke-width="1.5"/>
  <text x="240" y="103" text-anchor="middle" fill="#4a148c" font-size="10" font-weight="bold">Project + Pad</text>
  <text x="240" y="120" text-anchor="middle" fill="#4a148c" font-size="9">all → 4096-dim</text>
  <line x1="305" y1="107" x2="345" y2="107" stroke="#7c4dff" stroke-width="2" marker-end="url(#te)"/>
  <rect x="345" y="82" width="155" height="50" rx="7" fill="#e8f5e9" stroke="#34a853" stroke-width="2"/>
  <text x="422" y="100" text-anchor="middle" font-weight="bold" fill="#1b5e20" font-size="11">Joint Context</text>
  <text x="422" y="118" text-anchor="middle" fill="#555" font-size="10">[batch, ~154, 4096]</text>
  <text x="422" y="150" text-anchor="middle" fill="#555" font-size="9">→ MMDiT text tokens</text>
  <rect x="175" y="30" width="130" height="32" rx="5" fill="#fff3e0" stroke="#ff9800" stroke-width="1.5"/>
  <text x="240" y="44" text-anchor="middle" fill="#e65100" font-size="9" font-weight="bold">Pooled embeddings</text>
  <text x="240" y="56" text-anchor="middle" fill="#e65100" font-size="8">CLIP + OpenCLIP</text>
  <line x1="130" y1="44" x2="175" y2="44" stroke="#ff9800" stroke-width="1" stroke-dasharray="3,2"/>
  <line x1="130" y1="98" x2="175" y2="50" stroke="#ff9800" stroke-width="1" stroke-dasharray="3,2"/>
  <line x1="305" y1="46" x2="345" y2="46" stroke="#ff9800" stroke-width="1.5" marker-end="url(#te)"/>
  <rect x="345" y="28" width="155" height="38" rx="5" fill="#fff3e0" stroke="#ff9800" stroke-width="1.5"/>
  <text x="422" y="44" text-anchor="middle" fill="#e65100" font-size="10" font-weight="bold">adaLN conditioning</text>
  <text x="422" y="58" text-anchor="middle" fill="#e65100" font-size="9">+ timestep emb</text>
</svg>
</center>

**T5 can be dropped** (`text_encoder_3=None`) to save ~10 GB VRAM — quality degrades mainly on complex prompts and text rendering.

### 4.4 Training Innovations in SD 3.5

**Logit-Normal Timestep Sampling**

Instead of sampling $t$ uniformly as in DDPM:
- Sample $z \sim \mathcal{N}(0, 1)$, then $t = \sigma(z)$ where $\sigma$ is the sigmoid function
- This is the **logit-normal** distribution: $\text{logit}(t) \sim \mathcal{N}(0, 1)$

$$p(t) = \frac{1}{\sqrt{2\pi}\,t\,(1-t)} \exp\!\left(-\frac{(\text{logit}(t))^2}{2}\right)$$

**Effect**: concentrates training on intermediate timesteps ($t \approx 0.5$) where the denoising task is hardest (noise and signal compete). Early and late timesteps (easy cases) get less training weight.

**QK-Normalization**

Apply **RMSNorm** to $\mathbf{Q}$ and $\mathbf{K}$ before computing attention:

$$\hat{\mathbf{Q}} = \text{RMSNorm}(\mathbf{Q}), \quad \hat{\mathbf{K}} = \text{RMSNorm}(\mathbf{K})$$

$$\text{Attention} = \text{softmax}\!\left(\frac{\hat{\mathbf{Q}}\hat{\mathbf{K}}^\top}{\sqrt{d}}\right)\mathbf{V}$$

**Why**: Without QK-norm, attention logits can grow unboundedly as model width and sequence length increase, causing numerical overflow in mixed-precision (bf16/fp16) training. QK-norm bounds the magnitude and stabilizes training.

### 4.5 SD 3.5 Variants

| | **SD 3.5 Medium** | **SD 3.5 Large** |
|---|---|---|
| Parameters | 2.5B | 8B |
| Layers | 24 (MMDiT-X) | 38 (MMDiT) |
| Hidden size | 1536 | 2432 |
| Attention heads | 24 | 38 |
| joint_attention_dim | 4096 | 4096 |
| Patch size | 2×2 | 2×2 |
| VAE channels | 16 | 16 |
| VRAM (no T5, fp16) | ~5 GB | ~16+ GB |

**MMDiT-X** (Medium): adds self-attention within each modality stream alongside joint attention in the first 13 layers, improving coherence at smaller scale.

### 4.6 Running SD 3.5 Medium

Let's load SD 3.5 Medium (2.5B params) — fits comfortably on a T4 when we drop the T5 encoder.

> **Reference**: Esser et al. "Scaling Rectified Flow Transformers for High-Resolution Image Synthesis" (2024)

In [ ]:
from diffusers import StableDiffusion3Pipeline

pipe_sd3 = StableDiffusion3Pipeline.from_pretrained(
    "stabilityai/stable-diffusion-3.5-medium",
    text_encoder_3=None,  # drop T5 to save ~10GB VRAM
    tokenizer_3=None,
    torch_dtype=torch.float16,
).to("cuda")

print(f"Transformer params: {sum(p.numel() for p in pipe_sd3.transformer.parameters()) / 1e9:.1f}B")

generator = torch.Generator("cuda").manual_seed(SEED)
img_sd3 = pipe_sd3(
    PROMPT, negative_prompt=NEGATIVE, generator=generator,
    num_inference_steps=28,
).images[0]

plt.figure(figsize=(6, 6))
plt.imshow(img_sd3)
plt.axis("off")
plt.title("SD 3.5 Medium (1024×1024, 28 steps)")
plt.tight_layout()
plt.show()

### Rectified Flow needs fewer steps

One key advantage of rectified flow: straight trajectories need fewer discretization steps. Let's see how SD 3.5 quality varies with the number of steps:

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for i, steps in enumerate([5, 10, 20, 50]):
    gen = torch.Generator("cuda").manual_seed(SEED)
    img = pipe_sd3(PROMPT, generator=gen, num_inference_steps=steps).images[0]
    axes[i].imshow(img)
    axes[i].set_title(f"{steps} steps", fontsize=14)
    axes[i].axis("off")
plt.suptitle("SD 3.5: Quality vs Number of Steps (Rectified Flow)", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
del pipe_sd3
torch.cuda.empty_cache()
gc.collect()
print("SD 3.5 unloaded.")

## 5. FLUX — Hybrid Transformer Architecture

*Theory only — FLUX.1 has 12B parameters and does not fit on a Colab T4.*

**Background**: Created by **Black Forest Labs** (Robin Rombach et al. — the original Stable Diffusion creators). 12B parameters, rectified flow transformer.

| Variant | Distillation | Steps | License |
|---------|-------------|-------|---------|
| FLUX.1-schnell | Timestep-distilled | 1–4 | Apache 2.0 (open) |
| FLUX.1-dev | Guidance-distilled | 20–50 | Non-commercial (open weights) |
| FLUX.1-pro | Full teacher model | 20–50 | Proprietary, API-only |

### 5.1 Key Innovation — Double + Single Stream Blocks

MMDiT uses the same block type everywhere. FLUX introduces a **hybrid** architecture: different block types for early vs late layers.

**Architecture**: 19 double-stream blocks → 38 single-stream blocks (57 total). Hidden dim = 3072, 24 attention heads (128-dim each).

**Double-Stream Blocks** (first 19 layers) — modality-specific processing with joint attention:

<center>
<svg width="560" height="300" xmlns="http://www.w3.org/2000/svg" font-family="Arial, sans-serif" font-size="12">
  <defs><marker id="arr3" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#666"/></marker></defs>
  <rect x="1" y="1" width="558" height="298" rx="10" fill="none" stroke="#ccc" stroke-width="1.5" stroke-dasharray="6,3"/>
  <text x="280" y="22" text-anchor="middle" font-size="14" font-weight="bold" fill="#333">Double-Stream Block (×19)</text>
  <!-- Image stream -->
  <rect x="30" y="40" width="160" height="32" rx="6" fill="#e8f0fe" stroke="#4285f4" stroke-width="2"/>
  <text x="110" y="61" text-anchor="middle" font-weight="bold" fill="#1a73e8">Image Stream</text>
  <line x1="110" y1="72" x2="110" y2="92" stroke="#666" stroke-width="1.5" marker-end="url(#arr3)"/>
  <rect x="35" y="92" width="150" height="28" rx="5" fill="#fce4ec" stroke="#e91e63" stroke-width="1.5"/>
  <text x="110" y="111" text-anchor="middle" fill="#880e4f" font-size="11">Q/K/V<tspan font-size="9" baseline-shift="sub">img</tspan> (separate W)</text>
  <!-- Text stream -->
  <rect x="370" y="40" width="160" height="32" rx="6" fill="#fce8e6" stroke="#ea4335" stroke-width="2"/>
  <text x="450" y="61" text-anchor="middle" font-weight="bold" fill="#c5221f">Text Stream</text>
  <line x1="450" y1="72" x2="450" y2="92" stroke="#666" stroke-width="1.5" marker-end="url(#arr3)"/>
  <rect x="375" y="92" width="150" height="28" rx="5" fill="#fce4ec" stroke="#e91e63" stroke-width="1.5"/>
  <text x="450" y="111" text-anchor="middle" fill="#880e4f" font-size="11">Q/K/V<tspan font-size="9" baseline-shift="sub">txt</tspan> (separate W)</text>
  <!-- Arrows to joint attn -->
  <line x1="110" y1="120" x2="210" y2="145" stroke="#4285f4" stroke-width="2" marker-end="url(#arr3)"/>
  <line x1="450" y1="120" x2="350" y2="145" stroke="#ea4335" stroke-width="2" marker-end="url(#arr3)"/>
  <!-- Joint attention -->
  <rect x="180" y="140" width="200" height="38" rx="7" fill="#e8f5e9" stroke="#34a853" stroke-width="2.5"/>
  <text x="280" y="164" text-anchor="middle" font-weight="bold" fill="#1b5e20">Joint Attention</text>
  <!-- Arrows out -->
  <line x1="230" y1="178" x2="110" y2="205" stroke="#4285f4" stroke-width="2" marker-end="url(#arr3)"/>
  <line x1="330" y1="178" x2="450" y2="205" stroke="#ea4335" stroke-width="2" marker-end="url(#arr3)"/>
  <!-- MLPs -->
  <rect x="50" y="205" width="120" height="32" rx="6" fill="#e8f0fe" stroke="#4285f4" stroke-width="2"/>
  <text x="110" y="226" text-anchor="middle" font-weight="bold" fill="#1a73e8" font-size="11">MLP<tspan font-size="9" baseline-shift="sub">img</tspan></text>
  <rect x="390" y="205" width="120" height="32" rx="6" fill="#fce8e6" stroke="#ea4335" stroke-width="2"/>
  <text x="450" y="226" text-anchor="middle" font-weight="bold" fill="#c5221f" font-size="11">MLP<tspan font-size="9" baseline-shift="sub">txt</tspan></text>
  <!-- Output arrows -->
  <line x1="110" y1="237" x2="110" y2="262" stroke="#666" stroke-width="1.5" marker-end="url(#arr3)"/>
  <line x1="450" y1="237" x2="450" y2="262" stroke="#666" stroke-width="1.5" marker-end="url(#arr3)"/>
  <text x="110" y="278" text-anchor="middle" fill="#1a73e8" font-weight="bold" font-size="11">img out</text>
  <text x="450" y="278" text-anchor="middle" fill="#c5221f" font-weight="bold" font-size="11">txt out</text>
</svg>
</center>

Each stream has its **own Q/K/V projections and MLP** (separate weights), but attention itself is **joint** — Q/K/V from both streams are concatenated before the attention operation. Purpose: modality-specific feature extraction in early layers.

**Single-Stream Blocks** (last 38 layers) — unified processing:

<center>
<svg width="340" height="280" xmlns="http://www.w3.org/2000/svg" font-family="Arial, sans-serif" font-size="12">
  <defs><marker id="arr4" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#666"/></marker></defs>
  <rect x="1" y="1" width="338" height="278" rx="10" fill="none" stroke="#ccc" stroke-width="1.5" stroke-dasharray="6,3"/>
  <text x="170" y="22" text-anchor="middle" font-size="14" font-weight="bold" fill="#333">Single-Stream Block (×38)</text>
  <!-- Input: concat -->
  <rect x="50" y="42" width="240" height="32" rx="6" fill="#ede7f6" stroke="#7c4dff" stroke-width="2"/>
  <text x="170" y="63" text-anchor="middle" font-weight="bold" fill="#4a148c">[img tokens | txt tokens]</text>
  <line x1="170" y1="74" x2="170" y2="100" stroke="#666" stroke-width="1.5" marker-end="url(#arr4)"/>
  <!-- Self-Attn -->
  <rect x="60" y="100" width="220" height="38" rx="7" fill="#e8f5e9" stroke="#34a853" stroke-width="2.5"/>
  <text x="170" y="118" text-anchor="middle" font-weight="bold" fill="#1b5e20">Self-Attention</text>
  <text x="170" y="132" text-anchor="middle" fill="#555" font-size="10">(shared Q/K/V weights)</text>
  <line x1="170" y1="138" x2="170" y2="165" stroke="#666" stroke-width="1.5" marker-end="url(#arr4)"/>
  <!-- MLP -->
  <rect x="85" y="165" width="170" height="35" rx="6" fill="#f3e8fd" stroke="#9334e6" stroke-width="2"/>
  <text x="170" y="187" text-anchor="middle" font-weight="bold" fill="#7627bb">MLP (shared)</text>
  <line x1="170" y1="200" x2="170" y2="230" stroke="#666" stroke-width="1.5" marker-end="url(#arr4)"/>
  <!-- Output -->
  <rect x="50" y="230" width="240" height="32" rx="6" fill="#ede7f6" stroke="#7c4dff" stroke-width="2"/>
  <text x="170" y="251" text-anchor="middle" font-weight="bold" fill="#4a148c">[img | txt] output</text>
</svg>
</center>

Image and text tokens are concatenated into one sequence with **shared** attention and MLP weights. More parameter-efficient for deep multimodal fusion.

### 5.2 Rotary Positional Embeddings (RoPE)

Standard ViT/DiT uses absolute or learned positional embeddings added to the input. FLUX uses **3D axial RoPE** — a rotation in embedding space proportional to position.

For position $m$ and dimension pair $(2i, 2i+1)$:

$$\text{RoPE}(\mathbf{x}_m)_{2i} = x_{m,2i}\cos(m\theta_i) - x_{m,2i+1}\sin(m\theta_i)$$

$$\text{RoPE}(\mathbf{x}_m)_{2i+1} = x_{m,2i}\sin(m\theta_i) + x_{m,2i+1}\cos(m\theta_i)$$

where $\theta_i = 10000^{-2i/d}$.

**3D factorization** for images: the 128-dim head is split into:
- 16 dims for batch/temporal position
- 56 dims for height position
- 56 dims for width position

Each group gets its own RoPE frequencies computed independently, then concatenated.

**Benefits**: relative position encoding → better generalization to **unseen resolutions and aspect ratios** at inference time.

### 5.3 Guidance Distillation (FLUX.1-dev)

Standard classifier-free guidance requires **2× forward passes**:

$$\tilde{\mathbf{v}} = \mathbf{v}_{\text{uncond}} + w \cdot (\mathbf{v}_{\text{cond}} - \mathbf{v}_{\text{uncond}})$$

FLUX.1-dev distills this into a **single forward pass**:
- The guidance scale $w$ is provided as an **input to the model** via a learned embedding
- The model learns to approximate the CFG-guided output directly
- Result: **2× faster inference** with comparable quality

### 5.4 Timestep Distillation (FLUX.1-schnell)

Progressive distillation: halve the number of required steps iteratively:

$$1000 \to 500 \to 250 \to \ldots \to 4 \to 1$$

FLUX.1-schnell generates in **1–4 Euler steps** (vs 20–50 for FLUX.1-dev). Licensed under Apache 2.0 — fully open.

### 5.5 FLUX vs MMDiT (SD3) — Comparison

| | **MMDiT (SD3)** | **FLUX** |
|---|---|---|
| Block type | Uniform MMDiT blocks | Hybrid: double (19) + single (38) |
| Early layers | Joint attention (same as late) | Separate streams (modality-specific) |
| Late layers | Joint attention (same as early) | Merged streams (shared weights) |
| Position encoding | Learned absolute | 3D axial RoPE (relative) |
| Guidance | Standard CFG (2× cost) | Distilled guidance (1× cost) |
| Fast variant | — | Schnell (1–4 steps) |
| Text encoders | CLIP + OpenCLIP + T5 | T5 + CLIP |
| Parameters | 2.5–8B | 12B |

> **Reference**: Black Forest Labs "FLUX.1" (2024)

---
*Next: FluxKontext extends FLUX for image editing — replacing ControlNet and LoRA for many use cases.*

## 6. FluxKontext — Unified Generation and Editing

*Theory only — same 12B FLUX transformer, does not fit on Colab T4.*

### 6.1 Motivation

In **Seminar 10** we saw three approaches to controlled generation — each with limitations:
- **ControlNet**: needs edge/depth maps as explicit structural input
- **IP-Adapter**: needs a separate reference image encoder
- **LoRA**: needs 100+ training images and fine-tuning time

**FluxKontext** aims for a single model that handles generation AND editing via **in-context conditioning** — no auxiliary models, no fine-tuning.

### 6.2 Architecture — In-Context Image Conditioning

**Core mechanism**: sequence-level latent concatenation (not channel-level).

1. Input reference image → VAE encoder → **context latent tokens**
2. Random noise → **target latent tokens**
3. Context and target tokens are **concatenated along the sequence dimension**
4. The combined sequence is processed by the same 12B FLUX transformer
5. Output target tokens → VAE decoder → edited/generated image

<center>
<svg width="620" height="380" xmlns="http://www.w3.org/2000/svg" font-family="Arial, sans-serif" font-size="12">
  <defs><marker id="arr5" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#666"/></marker></defs>
  <!-- Input Image -->
  <rect x="30" y="20" width="150" height="45" rx="8" fill="#e8f0fe" stroke="#4285f4" stroke-width="2"/>
  <text x="105" y="38" text-anchor="middle" font-weight="bold" fill="#1a73e8">Input Image</text>
  <text x="105" y="55" text-anchor="middle" fill="#555" font-size="10">(reference)</text>
  <line x1="105" y1="65" x2="105" y2="90" stroke="#666" stroke-width="1.5" marker-end="url(#arr5)"/>
  <!-- VAE Encode -->
  <rect x="45" y="90" width="120" height="32" rx="6" fill="#e6f4ea" stroke="#34a853" stroke-width="1.5"/>
  <text x="105" y="111" text-anchor="middle" fill="#1b5e20" font-size="11">VAE Encode</text>
  <line x1="105" y1="122" x2="105" y2="148" stroke="#666" stroke-width="1.5" marker-end="url(#arr5)"/>
  <!-- Context tokens -->
  <rect x="30" y="148" width="150" height="32" rx="6" fill="#e8f0fe" stroke="#4285f4" stroke-width="2"/>
  <text x="105" y="169" text-anchor="middle" font-weight="bold" fill="#1a73e8" font-size="11">Context Tokens</text>
  <!-- Text Instruction -->
  <rect x="440" y="20" width="160" height="45" rx="8" fill="#fce8e6" stroke="#ea4335" stroke-width="2"/>
  <text x="520" y="38" text-anchor="middle" font-weight="bold" fill="#c5221f">Text Instruction</text>
  <text x="520" y="55" text-anchor="middle" fill="#555" font-size="10">"change to watercolor"</text>
  <line x1="520" y1="65" x2="520" y2="90" stroke="#666" stroke-width="1.5" marker-end="url(#arr5)"/>
  <!-- Text Encoders -->
  <rect x="450" y="90" width="140" height="32" rx="6" fill="#fce8e6" stroke="#ea4335" stroke-width="1.5"/>
  <text x="520" y="111" text-anchor="middle" fill="#c5221f" font-size="11">T5 + CLIP Encode</text>
  <line x1="520" y1="122" x2="520" y2="180" stroke="#666" stroke-width="1.5"/>
  <!-- Noise -->
  <rect x="230" y="130" width="150" height="32" rx="6" fill="#f3e8fd" stroke="#9334e6" stroke-width="2"/>
  <text x="305" y="151" text-anchor="middle" font-weight="bold" fill="#7627bb" font-size="11">Noise Target Tokens</text>
  <!-- Concat box -->
  <rect x="70" y="200" width="310" height="38" rx="7" fill="#ede7f6" stroke="#7c4dff" stroke-width="2"/>
  <text x="225" y="224" text-anchor="middle" font-weight="bold" fill="#4a148c">Concat along sequence dim</text>
  <line x1="105" y1="180" x2="160" y2="200" stroke="#4285f4" stroke-width="1.5" marker-end="url(#arr5)"/>
  <line x1="305" y1="162" x2="290" y2="200" stroke="#9334e6" stroke-width="1.5" marker-end="url(#arr5)"/>
  <!-- Arrow to transformer -->
  <line x1="225" y1="238" x2="310" y2="265" stroke="#666" stroke-width="2" marker-end="url(#arr5)"/>
  <line x1="520" y1="180" x2="400" y2="265" stroke="#ea4335" stroke-width="1.5" marker-end="url(#arr5)"/>
  <!-- Transformer -->
  <rect x="200" y="260" width="240" height="50" rx="10" fill="#fef7e0" stroke="#f9ab00" stroke-width="2.5"/>
  <text x="320" y="282" text-anchor="middle" font-weight="bold" font-size="13" fill="#e37400">FLUX Transformer (12B)</text>
  <text x="320" y="300" text-anchor="middle" fill="#555" font-size="10">19 Double + 38 Single Stream</text>
  <!-- Arrow to VAE Decode -->
  <line x1="320" y1="310" x2="320" y2="335" stroke="#666" stroke-width="2" marker-end="url(#arr5)"/>
  <!-- VAE Decode -->
  <rect x="260" y="335" width="120" height="32" rx="6" fill="#e6f4ea" stroke="#34a853" stroke-width="1.5"/>
  <text x="320" y="356" text-anchor="middle" fill="#1b5e20" font-weight="bold" font-size="11">VAE Decode</text>
</svg>
</center>

**3D RoPE with virtual time offset**: context tokens receive a constant offset in the "time" dimension of the 3D RoPE triplet $(t, h, w)$, cleanly separating them from target tokens in positional space while preserving internal spatial structure.

### 6.3 Comparison with Seminar 10 Methods

| Method | Needs | Training? | Zero-shot? | Structural control |
|--------|-------|-----------|------------|-------------------|
| ControlNet | Edge/depth maps | Pre-trained adapters | No (needs maps) | High (explicit) |
| IP-Adapter | Reference image | Pre-trained adapters | Yes | Medium (style) |
| LoRA | 100+ image dataset | Fine-tuning | No | Low (global) |
| **FluxKontext** | Input image + text | **None** | **Yes** | High (instruction) |

**Capabilities**: zero-shot style transfer, object-level editing, character consistency across generations, multi-image context input.

> **Reference**: Xiao et al. "FLUX.1 Kontext" (2025)

## 7. NanoBanana — Efficient MMDiT for Production

*Theory only — no open weights, API-only access.*

### 7.1 Background

**NanoBanana** is Google DeepMind's image generation model (anonymous codename on LMArena), integrated natively with **Gemini**. Built on the MMDiT architecture (similar foundation as SD3), it focuses on production efficiency and quality.

### 7.2 Key Achievements

- **60% faster** generation than conventional diffusion models
- **94% character accuracy** in text rendering (vs DALL-E 3 at ~78%) — critical for real-world applications
- **#1 on LMArena** for both text-to-image and image editing (ELO ~1362 vs DALL-E 3: ~1187)
- **Multi-turn conversational editing**: generate → refine → refine (natively within Gemini chat)

### 7.3 Efficiency Techniques (publicly known)

- **Latent representation anchoring**: for iterative refinement — avoid full re-noising between edit rounds
- **Optimized transformer blocks**: architectural modifications for inference speed
- **LLM integration**: Gemini understands complex prompts, decomposes them, and drives the generation pipeline
- **Production optimization**: quantization and JAX/XLA compilation for serving at scale

### 7.4 API Access

No open weights — available only through the Gemini API:

```python
# Conceptual — NOT runnable (requires API key + quota)
from google import genai

client = genai.Client(api_key="...")
response = client.models.generate_images(
    model="gemini-2.0-flash-preview-image-generation",
    prompt="A photograph of an astronaut riding a horse on Mars",
)
# response.generated_images[0].image contains the PIL Image
```

### 7.5 Open vs Closed Models

| | **Open** (SD, FLUX) | **Closed** (NanoBanana, DALL-E, Midjourney) |
|---|---|---|
| Control | Full — modify weights, LoRA, self-host | None — API only |
| Customization | LoRA, DreamBooth, ControlNet | Prompt engineering only |
| Cost | GPU time (self-hosted) | Per-API-call pricing |
| Quality ceiling | FLUX ≈ older closed models | Generally higher (for now) |
| Community | Huge open-source ecosystem | Walled garden |
| Reproducibility | Full | Non-deterministic APIs |

**Trend**: the gap is narrowing. FLUX matches or exceeds DALL-E 3 on many benchmarks. But NanoBanana / Gemini push the frontier further — especially on text rendering and multi-turn editing.

---
*Next: let's summarize the entire evolution in one table.*

## 8. Architecture Evolution Summary

### 8.1 The Big Table

| Feature | **SD 1.5** (2022) | **SDXL** (2023) | **SD 3.5** (2024) | **FLUX.1** (2024) |
|---------|:-:|:-:|:-:|:-:|
| **Backbone** | UNet | UNet (larger) | MMDiT | Hybrid Transformer |
| **Parameters** | 860M | ~3.5B | 2–8B | 12B |
| **Text encoders** | CLIP ViT-L | CLIP + OpenCLIP | CLIP + OpenCLIP + T5 | T5 + CLIP |
| **Context dim** | 768 | 2048 | 4096 | 4096 |
| **Resolution** | 512² | 1024² | 1024² | 1024²+ |
| **Training** | DDPM ($\boldsymbol\epsilon$-pred) | DDPM ($\boldsymbol\epsilon$-pred) | Rectified Flow ($\mathbf{v}$-pred) | Rectified Flow ($\mathbf{v}$-pred) |
| **Min steps** | ~50 | ~30 | ~20 | ~4 (schnell) |
| **Attention** | Cross-Attn only | Cross-Attn only | Joint Self-Attn (MMDiT) | Double + Single Stream |
| **Pos. encoding** | Learned | Learned + Fourier | Learned | 3D axial RoPE |
| **Guidance** | Standard CFG | Standard CFG | Standard CFG | Distilled (1× cost) |
| **VAE channels** | 4 | 4 | 16 | 16 |

### Visual Evolution Timeline

<center>
<svg width="660" height="200" xmlns="http://www.w3.org/2000/svg" font-family="Arial, sans-serif" font-size="11">
  <!-- Timeline bar -->
  <line x1="40" y1="90" x2="620" y2="90" stroke="#ddd" stroke-width="3"/>
  <!-- SD 1.5 -->
  <circle cx="80" cy="90" r="18" fill="#fef7e0" stroke="#f9ab00" stroke-width="2.5"/>
  <text x="80" y="94" text-anchor="middle" font-weight="bold" fill="#e37400" font-size="9">1.5</text>
  <text x="80" y="126" text-anchor="middle" font-weight="bold" fill="#444" font-size="11">SD 1.5</text>
  <text x="80" y="140" text-anchor="middle" fill="#888" font-size="9">UNet 860M</text>
  <text x="80" y="153" text-anchor="middle" fill="#888" font-size="9">DDPM, 1 CLIP</text>
  <text x="80" y="166" text-anchor="middle" fill="#888" font-size="9">512², 50 steps</text>
  <text x="80" y="70" text-anchor="middle" fill="#aaa" font-size="10">2022</text>
  <!-- SDXL -->
  <circle cx="230" cy="90" r="22" fill="#fef7e0" stroke="#f9ab00" stroke-width="2.5"/>
  <text x="230" y="94" text-anchor="middle" font-weight="bold" fill="#e37400" font-size="9">XL</text>
  <text x="230" y="126" text-anchor="middle" font-weight="bold" fill="#444" font-size="11">SDXL</text>
  <text x="230" y="140" text-anchor="middle" fill="#888" font-size="9">UNet 3.5B</text>
  <text x="230" y="153" text-anchor="middle" fill="#888" font-size="9">DDPM, 2 CLIPs</text>
  <text x="230" y="166" text-anchor="middle" fill="#888" font-size="9">1024², 30 steps</text>
  <text x="230" y="66" text-anchor="middle" fill="#aaa" font-size="10">2023</text>
  <!-- SD 3.5 -->
  <circle cx="400" cy="90" r="26" fill="#e8f5e9" stroke="#34a853" stroke-width="2.5"/>
  <text x="400" y="94" text-anchor="middle" font-weight="bold" fill="#1b5e20" font-size="9">3.5</text>
  <text x="400" y="126" text-anchor="middle" font-weight="bold" fill="#444" font-size="11">SD 3.5</text>
  <text x="400" y="140" text-anchor="middle" fill="#888" font-size="9">MMDiT 2.5–8B</text>
  <text x="400" y="153" text-anchor="middle" fill="#34a853" font-size="9" font-weight="bold">Rect. Flow, 3 enc</text>
  <text x="400" y="166" text-anchor="middle" fill="#888" font-size="9">1024², 20 steps</text>
  <text x="400" y="62" text-anchor="middle" fill="#aaa" font-size="10">2024</text>
  <!-- FLUX -->
  <circle cx="570" cy="90" r="30" fill="#e8f0fe" stroke="#4285f4" stroke-width="2.5"/>
  <text x="570" y="88" text-anchor="middle" font-weight="bold" fill="#1a73e8" font-size="9">FLUX</text>
  <text x="570" y="100" text-anchor="middle" fill="#1a73e8" font-size="8">.1</text>
  <text x="570" y="130" text-anchor="middle" font-weight="bold" fill="#444" font-size="11">FLUX.1</text>
  <text x="570" y="144" text-anchor="middle" fill="#888" font-size="9">Hybrid 12B</text>
  <text x="570" y="157" text-anchor="middle" fill="#4285f4" font-size="9" font-weight="bold">Distilled, RoPE</text>
  <text x="570" y="170" text-anchor="middle" fill="#888" font-size="9">1024²+, 4 steps</text>
  <text x="570" y="58" text-anchor="middle" fill="#aaa" font-size="10">2024</text>
  <!-- Arrows between -->
  <line x1="102" y1="90" x2="206" y2="90" stroke="#f9ab00" stroke-width="2" marker-end="url(#te)"/>
  <line x1="254" y1="90" x2="372" y2="90" stroke="#34a853" stroke-width="2" marker-end="url(#te)"/>
  <line x1="428" y1="90" x2="538" y2="90" stroke="#4285f4" stroke-width="2" marker-end="url(#te)"/>
  <!-- Labels on arrows -->
  <text x="155" y="82" text-anchor="middle" fill="#f9ab00" font-size="8" font-weight="bold">scale up</text>
  <text x="318" y="82" text-anchor="middle" fill="#34a853" font-size="8" font-weight="bold">UNet→MMDiT</text>
  <text x="485" y="82" text-anchor="middle" fill="#4285f4" font-size="8" font-weight="bold">hybrid+distill</text>
  <!-- Size growth indicator -->
  <text x="350" y="196" text-anchor="middle" fill="#aaa" font-size="10" font-style="italic">Circle size ~ parameter count (860M → 12B, 14× growth)</text>
</svg>
</center>

### 8.2 Five Axes of Evolution

1. **Architecture**: UNet → DiT → MMDiT → Hybrid Transformer (double + single stream)
2. **Training paradigm**: DDPM noise prediction → Flow Matching / Rectified Flow velocity prediction
3. **Text understanding**: 1 encoder (768-dim) → 3 encoders (4096-dim) with T5 for complex text
4. **Scale**: 860M → 12B parameters (14× growth in 2 years)
5. **Capabilities**: text-to-image only → unified generation + in-context editing (FluxKontext)

### 8.3 What Stayed the Same

- **VAE** for latent space compression (all models use a similar autoencoder)
- **Classifier-free guidance** principle (even if distilled in FLUX)
- **Latent space** at ~1/8 spatial resolution (4–16 channels)
- **HuggingFace `diffusers`** as the standard interface for all open models

---
*Next: let's compare SD 1.5 and SD 3.5 side by side on the same prompts.*

## 9. Practical Comparison — SD 1.5 vs SD 3.5

Let's compare the two runnable models on multiple prompts to see how the evolution affects real outputs. We test:
- **Composition**: astronaut on horse (multi-object scene)
- **Text rendering**: text on a coffee mug (historically weak for diffusion models)
- **Spatial reasoning**: cat sitting on top of dog (relative positioning)
- **Style transfer**: Van Gogh + cyberpunk (complex style combination)

In [ ]:
prompts = [
    "A photograph of an astronaut riding a horse on Mars, detailed, 8k",
    "A coffee mug with the text 'Deep Generative Models' written on it",
    "A cat sitting on top of a dog, in a park, photorealistic",
    "An oil painting of a cyberpunk city at night in the style of Van Gogh",
]

# --- Generate with SD 1.5 ---
pipe_sd15 = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16, safety_checker=None,
).to("cuda")

results_sd15 = []
for prompt in tqdm(prompts, desc="SD 1.5"):
    gen = torch.Generator("cuda").manual_seed(SEED)
    img = pipe_sd15(prompt, negative_prompt=NEGATIVE, generator=gen,
                    num_inference_steps=50).images[0]
    results_sd15.append(img)

del pipe_sd15
torch.cuda.empty_cache()
gc.collect()
print("SD 1.5 done, unloaded.")

In [ ]:
# --- Generate with SD 3.5 Medium ---
pipe_sd3 = StableDiffusion3Pipeline.from_pretrained(
    "stabilityai/stable-diffusion-3.5-medium",
    text_encoder_3=None, tokenizer_3=None,
    torch_dtype=torch.float16,
).to("cuda")

results_sd3 = []
for prompt in tqdm(prompts, desc="SD 3.5"):
    gen = torch.Generator("cuda").manual_seed(SEED)
    img = pipe_sd3(prompt, negative_prompt=NEGATIVE, generator=gen,
                   num_inference_steps=28).images[0]
    results_sd3.append(img)

del pipe_sd3
torch.cuda.empty_cache()
gc.collect()
print("SD 3.5 done, unloaded.")

In [ ]:
# --- Side-by-side comparison grid ---
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

short_labels = ["Astronaut", "Text on mug", "Cat on dog", "Van Gogh cyberpunk"]

for j in range(4):
    axes[0, j].imshow(results_sd15[j])
    axes[0, j].set_title(f"SD 1.5", fontsize=12)
    axes[0, j].set_xlabel(short_labels[j], fontsize=10)
    axes[0, j].axis("off")

    axes[1, j].imshow(results_sd3[j])
    axes[1, j].set_title(f"SD 3.5", fontsize=12)
    axes[1, j].axis("off")

plt.suptitle(
    "SD 1.5 (UNet + DDPM, 50 steps) vs SD 3.5 (MMDiT + Rectified Flow, 28 steps)",
    fontsize=14,
)
plt.tight_layout()
plt.show()

### Observations

**What to look for in the comparison:**

- **Text rendering** (mug prompt): SD 3.5 should produce more readable text thanks to the richer text encoders (especially T5 in the full model). SD 1.5 typically produces illegible text.
- **Composition** (astronaut): SD 3.5 produces more coherent scene composition with better spatial relationships.
- **Spatial reasoning** (cat on dog): joint self-attention (MMDiT) helps the model understand "on top of" better than cross-attention alone.
- **Style mixing** (Van Gogh + cyberpunk): both models handle this, but SD 3.5 at 1024² has more room for detail.
- **Speed**: SD 1.5 uses 50 steps, SD 3.5 uses only 28 steps — rectified flow's straight paths converge faster.

---
*Next: let's visualize the denoising process itself to see HOW the two sampling strategies differ.*

## 10. Sampling Comparison — DDPM vs Rectified Flow

### The Two Sampling Strategies

**DDPM sampling** (SD 1.5) — iterative noise removal with a complex schedule:

$$\mathbf{x}_{t-1} = \frac{1}{\sqrt{\alpha_t}}\left(\mathbf{x}_t - \frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\boldsymbol\epsilon_\theta(\mathbf{x}_t, t)\right) + \sigma_t\,\mathbf{z}$$

**Rectified Flow sampling** (SD 3.5) — simple Euler ODE integration along straight paths:

$$\mathbf{x}_{t+\Delta t} = \mathbf{x}_t + \Delta t \cdot \mathbf{v}_\theta(\mathbf{x}_t, t)$$

Let's capture intermediate latents during generation to visualize how each model progressively denoises the image.

### Sampling Side by Side

<center>
<svg width="620" height="190" xmlns="http://www.w3.org/2000/svg" font-family="Arial, sans-serif" font-size="11">
  <defs><marker id="sa" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#666"/></marker></defs>
  <!-- DDPM -->
  <text x="150" y="16" text-anchor="middle" font-size="13" font-weight="bold" fill="#ea4335">DDPM (SD 1.5)</text>
  <rect x="15" y="28" width="70" height="32" rx="5" fill="#f3e8fd" stroke="#9334e6" stroke-width="1.5"/>
  <text x="50" y="49" text-anchor="middle" fill="#7627bb" font-size="10" font-weight="bold">x_T ~ N</text>
  <line x1="85" y1="44" x2="105" y2="44" stroke="#666" stroke-width="1" marker-end="url(#sa)"/>
  <rect x="105" y="30" width="28" height="28" rx="4" fill="#fce8e6" stroke="#ea4335" stroke-width="1"/>
  <text x="119" y="49" text-anchor="middle" fill="#c5221f" font-size="9">eps</text>
  <line x1="133" y1="44" x2="145" y2="44" stroke="#666" stroke-width="1" marker-end="url(#sa)"/>
  <rect x="145" y="30" width="28" height="28" rx="4" fill="#fce8e6" stroke="#ea4335" stroke-width="1"/>
  <text x="159" y="49" text-anchor="middle" fill="#c5221f" font-size="9">eps</text>
  <line x1="173" y1="44" x2="180" y2="44" stroke="#666" stroke-width="1"/>
  <text x="190" y="48" fill="#888" font-size="12">...</text>
  <line x1="200" y1="44" x2="208" y2="44" stroke="#666" stroke-width="1"/>
  <rect x="208" y="30" width="28" height="28" rx="4" fill="#fce8e6" stroke="#ea4335" stroke-width="1"/>
  <text x="222" y="49" text-anchor="middle" fill="#c5221f" font-size="9">eps</text>
  <line x1="236" y1="44" x2="256" y2="44" stroke="#666" stroke-width="1" marker-end="url(#sa)"/>
  <rect x="256" y="28" width="40" height="32" rx="5" fill="#e6f4ea" stroke="#34a853" stroke-width="1.5"/>
  <text x="276" y="49" text-anchor="middle" fill="#1b5e20" font-size="10" font-weight="bold">x_0</text>
  <text x="150" y="78" text-anchor="middle" fill="#c5221f" font-size="10">50 noise-prediction steps, complex schedule</text>
  <!-- Divider -->
  <line x1="310" y1="10" x2="310" y2="170" stroke="#ddd" stroke-width="1" stroke-dasharray="4,3"/>
  <!-- Rectified Flow -->
  <text x="470" y="16" text-anchor="middle" font-size="13" font-weight="bold" fill="#34a853">Rectified Flow (SD 3.5)</text>
  <rect x="335" y="28" width="70" height="32" rx="5" fill="#f3e8fd" stroke="#9334e6" stroke-width="1.5"/>
  <text x="370" y="49" text-anchor="middle" fill="#7627bb" font-size="10" font-weight="bold">x_0 ~ N</text>
  <line x1="405" y1="44" x2="425" y2="44" stroke="#34a853" stroke-width="1.5" marker-end="url(#sa)"/>
  <rect x="425" y="30" width="28" height="28" rx="4" fill="#e8f5e9" stroke="#34a853" stroke-width="1.5"/>
  <text x="439" y="48" text-anchor="middle" fill="#1b5e20" font-size="9">v</text>
  <line x1="453" y1="44" x2="470" y2="44" stroke="#34a853" stroke-width="1.5" marker-end="url(#sa)"/>
  <rect x="470" y="30" width="28" height="28" rx="4" fill="#e8f5e9" stroke="#34a853" stroke-width="1.5"/>
  <text x="484" y="48" text-anchor="middle" fill="#1b5e20" font-size="9">v</text>
  <line x1="498" y1="44" x2="505" y2="44" stroke="#34a853" stroke-width="1"/>
  <text x="512" y="48" fill="#888" font-size="12">...</text>
  <line x1="520" y1="44" x2="530" y2="44" stroke="#34a853" stroke-width="1"/>
  <rect x="530" y="30" width="28" height="28" rx="4" fill="#e8f5e9" stroke="#34a853" stroke-width="1.5"/>
  <text x="544" y="48" text-anchor="middle" fill="#1b5e20" font-size="9">v</text>
  <line x1="558" y1="44" x2="575" y2="44" stroke="#34a853" stroke-width="1.5" marker-end="url(#sa)"/>
  <rect x="575" y="28" width="40" height="32" rx="5" fill="#e6f4ea" stroke="#34a853" stroke-width="2"/>
  <text x="595" y="49" text-anchor="middle" fill="#1b5e20" font-size="10" font-weight="bold">x_1</text>
  <text x="470" y="78" text-anchor="middle" fill="#1b5e20" font-size="10">28 velocity steps, simple Euler</text>
  <!-- Formulas -->
  <rect x="15" y="95" width="280" height="65" rx="6" fill="#fff8f8" stroke="#fcc" stroke-width="1"/>
  <text x="155" y="115" text-anchor="middle" fill="#333" font-size="10" font-family="serif, Times">x<tspan font-size="8" baseline-shift="sub">t-1</tspan> = (1/sqrt(a)) (x<tspan font-size="8" baseline-shift="sub">t</tspan> - B*eps) + sigma*z</text>
  <text x="155" y="135" text-anchor="middle" fill="#888" font-size="9">complex schedule: alpha, beta, sigma</text>
  <text x="155" y="150" text-anchor="middle" fill="#ea4335" font-size="9" font-weight="bold">SDE / DDIM ODE</text>
  <rect x="335" y="95" width="280" height="65" rx="6" fill="#f5fff5" stroke="#cfc" stroke-width="1"/>
  <text x="475" y="115" text-anchor="middle" fill="#333" font-size="10" font-family="serif, Times">x<tspan font-size="8" baseline-shift="sub">t+dt</tspan> = x<tspan font-size="8" baseline-shift="sub">t</tspan> + dt * v(x<tspan font-size="8" baseline-shift="sub">t</tspan>, t)</text>
  <text x="475" y="135" text-anchor="middle" fill="#888" font-size="9">simple linear: t in [0, 1]</text>
  <text x="475" y="150" text-anchor="middle" fill="#34a853" font-size="9" font-weight="bold">Euler ODE</text>
</svg>
</center>

### Sampling Side by Side

<center>
<svg width="620" height="190" xmlns="http://www.w3.org/2000/svg" font-family="Arial, sans-serif" font-size="11">
  <defs><marker id="sa" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#666"/></marker></defs>
  <!-- DDPM -->
  <text x="150" y="16" text-anchor="middle" font-size="13" font-weight="bold" fill="#ea4335">DDPM (SD 1.5)</text>
  <rect x="15" y="28" width="70" height="32" rx="5" fill="#f3e8fd" stroke="#9334e6" stroke-width="1.5"/>
  <text x="50" y="49" text-anchor="middle" fill="#7627bb" font-size="10" font-weight="bold">x_T ~ N</text>
  <line x1="85" y1="44" x2="105" y2="44" stroke="#666" stroke-width="1" marker-end="url(#sa)"/>
  <rect x="105" y="30" width="28" height="28" rx="4" fill="#fce8e6" stroke="#ea4335" stroke-width="1"/>
  <text x="119" y="49" text-anchor="middle" fill="#c5221f" font-size="9">eps</text>
  <line x1="133" y1="44" x2="145" y2="44" stroke="#666" stroke-width="1" marker-end="url(#sa)"/>
  <rect x="145" y="30" width="28" height="28" rx="4" fill="#fce8e6" stroke="#ea4335" stroke-width="1"/>
  <text x="159" y="49" text-anchor="middle" fill="#c5221f" font-size="9">eps</text>
  <line x1="173" y1="44" x2="180" y2="44" stroke="#666" stroke-width="1"/>
  <text x="190" y="48" fill="#888" font-size="12">...</text>
  <line x1="200" y1="44" x2="208" y2="44" stroke="#666" stroke-width="1"/>
  <rect x="208" y="30" width="28" height="28" rx="4" fill="#fce8e6" stroke="#ea4335" stroke-width="1"/>
  <text x="222" y="49" text-anchor="middle" fill="#c5221f" font-size="9">eps</text>
  <line x1="236" y1="44" x2="256" y2="44" stroke="#666" stroke-width="1" marker-end="url(#sa)"/>
  <rect x="256" y="28" width="40" height="32" rx="5" fill="#e6f4ea" stroke="#34a853" stroke-width="1.5"/>
  <text x="276" y="49" text-anchor="middle" fill="#1b5e20" font-size="10" font-weight="bold">x_0</text>
  <text x="150" y="78" text-anchor="middle" fill="#c5221f" font-size="10">50 noise-prediction steps, complex schedule</text>
  <!-- Divider -->
  <line x1="310" y1="10" x2="310" y2="170" stroke="#ddd" stroke-width="1" stroke-dasharray="4,3"/>
  <!-- Rectified Flow -->
  <text x="470" y="16" text-anchor="middle" font-size="13" font-weight="bold" fill="#34a853">Rectified Flow (SD 3.5)</text>
  <rect x="335" y="28" width="70" height="32" rx="5" fill="#f3e8fd" stroke="#9334e6" stroke-width="1.5"/>
  <text x="370" y="49" text-anchor="middle" fill="#7627bb" font-size="10" font-weight="bold">x_0 ~ N</text>
  <line x1="405" y1="44" x2="425" y2="44" stroke="#34a853" stroke-width="1.5" marker-end="url(#sa)"/>
  <rect x="425" y="30" width="28" height="28" rx="4" fill="#e8f5e9" stroke="#34a853" stroke-width="1.5"/>
  <text x="439" y="48" text-anchor="middle" fill="#1b5e20" font-size="9">v</text>
  <line x1="453" y1="44" x2="470" y2="44" stroke="#34a853" stroke-width="1.5" marker-end="url(#sa)"/>
  <rect x="470" y="30" width="28" height="28" rx="4" fill="#e8f5e9" stroke="#34a853" stroke-width="1.5"/>
  <text x="484" y="48" text-anchor="middle" fill="#1b5e20" font-size="9">v</text>
  <line x1="498" y1="44" x2="505" y2="44" stroke="#34a853" stroke-width="1"/>
  <text x="512" y="48" fill="#888" font-size="12">...</text>
  <line x1="520" y1="44" x2="530" y2="44" stroke="#34a853" stroke-width="1"/>
  <rect x="530" y="30" width="28" height="28" rx="4" fill="#e8f5e9" stroke="#34a853" stroke-width="1.5"/>
  <text x="544" y="48" text-anchor="middle" fill="#1b5e20" font-size="9">v</text>
  <line x1="558" y1="44" x2="575" y2="44" stroke="#34a853" stroke-width="1.5" marker-end="url(#sa)"/>
  <rect x="575" y="28" width="40" height="32" rx="5" fill="#e6f4ea" stroke="#34a853" stroke-width="2"/>
  <text x="595" y="49" text-anchor="middle" fill="#1b5e20" font-size="10" font-weight="bold">x_1</text>
  <text x="470" y="78" text-anchor="middle" fill="#1b5e20" font-size="10">28 velocity steps, simple Euler</text>
  <!-- Formulas -->
  <rect x="15" y="95" width="280" height="65" rx="6" fill="#fff8f8" stroke="#fcc" stroke-width="1"/>
  <text x="155" y="115" text-anchor="middle" fill="#333" font-size="10" font-family="serif, Times">x<tspan font-size="8" baseline-shift="sub">t-1</tspan> = (1/sqrt(a)) (x<tspan font-size="8" baseline-shift="sub">t</tspan> - B*eps) + sigma*z</text>
  <text x="155" y="135" text-anchor="middle" fill="#888" font-size="9">complex schedule: alpha, beta, sigma</text>
  <text x="155" y="150" text-anchor="middle" fill="#ea4335" font-size="9" font-weight="bold">SDE / DDIM ODE</text>
  <rect x="335" y="95" width="280" height="65" rx="6" fill="#f5fff5" stroke="#cfc" stroke-width="1"/>
  <text x="475" y="115" text-anchor="middle" fill="#333" font-size="10" font-family="serif, Times">x<tspan font-size="8" baseline-shift="sub">t+dt</tspan> = x<tspan font-size="8" baseline-shift="sub">t</tspan> + dt * v(x<tspan font-size="8" baseline-shift="sub">t</tspan>, t)</text>
  <text x="475" y="135" text-anchor="middle" fill="#888" font-size="9">simple linear: t in [0, 1]</text>
  <text x="475" y="150" text-anchor="middle" fill="#34a853" font-size="9" font-weight="bold">Euler ODE</text>
</svg>
</center>

In [ ]:
# --- Capture intermediate latents for SD 1.5 ---
pipe_sd15 = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16, safety_checker=None,
).to("cuda")

intermediates_sd15 = []

def callback_sd15(pipe, step, timestep, kwargs):
    intermediates_sd15.append(kwargs["latents"].detach().cpu())
    return kwargs

gen = torch.Generator("cuda").manual_seed(SEED)
_ = pipe_sd15(
    PROMPT, negative_prompt=NEGATIVE, generator=gen,
    num_inference_steps=50, callback_on_step_end=callback_sd15,
)

# Decode selected intermediate latents to images
vae_sd15 = pipe_sd15.vae
step_indices_sd15 = [0, 4, 9, 19, 34, 49]  # steps 1, 5, 10, 20, 35, 50
decoded_sd15 = []
for idx in step_indices_sd15:
    with torch.no_grad():
        latent = intermediates_sd15[idx].to("cuda", dtype=torch.float16)
        image = vae_sd15.decode(latent / vae_sd15.config.scaling_factor).sample
        image = (image / 2 + 0.5).clamp(0, 1).cpu().permute(0, 2, 3, 1).numpy()[0]
        decoded_sd15.append(image)

del pipe_sd15, vae_sd15, intermediates_sd15
torch.cuda.empty_cache()
gc.collect()
print(f"SD 1.5: captured {len(decoded_sd15)} intermediate images.")

In [ ]:
# --- Capture intermediate latents for SD 3.5 ---
pipe_sd3 = StableDiffusion3Pipeline.from_pretrained(
    "stabilityai/stable-diffusion-3.5-medium",
    text_encoder_3=None, tokenizer_3=None,
    torch_dtype=torch.float16,
).to("cuda")

intermediates_sd3 = []

def callback_sd3(pipe, step, timestep, kwargs):
    intermediates_sd3.append(kwargs["latents"].detach().cpu())
    return kwargs

gen = torch.Generator("cuda").manual_seed(SEED)
_ = pipe_sd3(
    PROMPT, negative_prompt=NEGATIVE, generator=gen,
    num_inference_steps=28, callback_on_step_end=callback_sd3,
)

# Decode selected intermediate latents to images
# SD3 VAE decode: latent / scaling_factor + shift_factor, then vae.decode()
vae_sd3 = pipe_sd3.vae
step_indices_sd3 = [0, 2, 5, 10, 18, 27]  # ~evenly spaced across 28 steps
decoded_sd3 = []
for idx in step_indices_sd3:
    with torch.no_grad():
        latent = intermediates_sd3[idx].to("cuda", dtype=torch.float16)
        latent_for_vae = latent / vae_sd3.config.scaling_factor + vae_sd3.config.shift_factor
        image = vae_sd3.decode(latent_for_vae).sample
        image = (image / 2 + 0.5).clamp(0, 1).cpu().permute(0, 2, 3, 1).numpy()[0]
        decoded_sd3.append(image)

del pipe_sd3, vae_sd3, intermediates_sd3
torch.cuda.empty_cache()
gc.collect()
print(f"SD 3.5: captured {len(decoded_sd3)} intermediate images.")

In [ ]:
# --- Visualize denoising progression ---
fig, axes = plt.subplots(2, 6, figsize=(24, 8))

sd15_labels = ["Step 1/50", "Step 5/50", "Step 10/50", "Step 20/50", "Step 35/50", "Step 50/50"]
sd3_labels = ["Step 1/28", "Step 3/28", "Step 6/28", "Step 11/28", "Step 19/28", "Step 28/28"]

for j in range(6):
    axes[0, j].imshow(decoded_sd15[j])
    axes[0, j].set_title(sd15_labels[j], fontsize=10)
    axes[0, j].axis("off")

    axes[1, j].imshow(decoded_sd3[j])
    axes[1, j].set_title(sd3_labels[j], fontsize=10)
    axes[1, j].axis("off")

axes[0, 0].set_ylabel("SD 1.5\n(DDPM)", fontsize=12, rotation=0, labelpad=60, va="center")
axes[1, 0].set_ylabel("SD 3.5\n(Rect. Flow)", fontsize=12, rotation=0, labelpad=60, va="center")

plt.suptitle("Denoising Progression: DDPM (50 steps) vs Rectified Flow (28 steps)", fontsize=14)
plt.tight_layout()
plt.show()

### What to observe

- **Convergence speed**: Rectified Flow (SD 3.5) should produce recognizable structure faster relative to its total step count. The straight-line trajectories mean the model makes more progress per step.
- **Noise pattern**: DDPM (SD 1.5) noise gradually reveals structure through curved denoising. Rectified Flow linearly interpolates from noise to the target image.
- **Final quality**: Despite using fewer steps (28 vs 50), SD 3.5 produces higher-resolution images with better text-image alignment thanks to the MMDiT architecture and triple text encoders.

## 11. Discussion & Takeaways

### When to Use Which Model

| Use case | Best choice | Why |
|----------|-------------|-----|
| Quick prototyping | SD 1.5 | Lightweight, fast, huge community, runs anywhere |
| Production quality | FLUX.1-schnell | Best open quality, 4 steps, Apache 2.0 |
| Custom fine-tuning | SD 1.5 / SDXL | Mature LoRA/DreamBooth ecosystem |
| Text-heavy images | SD 3.5 or NanoBanana | T5 encoder for accurate typography |
| Image editing | FluxKontext | Zero-shot instruction-based, no adapters needed |
| Research | SD 3.5 | Open weights, well-documented, reproducible |
| Highest possible quality | NanoBanana (Gemini API) | State-of-the-art, but closed and API-only |

### Connection to Course

| This seminar | Builds on |
|---|---|
| Section 1 (DDPM → FM theory) | Lectures 11–12 (Flow Matching, Conditional FM) |
| Section 4 (MMDiT architecture) | Lectures 9–10 (ODEs, probability flow) |
| Sections 2, 9 (SD 1.5 code) | Seminar 9 (UNet from scratch, SD playground) |
| Section 6 (FluxKontext) | Seminar 10 (ControlNet, IP-Adapter, LoRA) |

### What's Next

- **Lectures 13–14**: Discrete diffusion — extending the diffusion framework to discrete data (text, graphs)
- **Masked diffusion language models**: applying diffusion to token sequences

### References

1. Rombach et al. "High-Resolution Image Synthesis with Latent Diffusion Models" (2022)
2. Podell et al. "SDXL: Improving Latent Diffusion Models for High-Resolution Image Synthesis" (2023)
3. Peebles & Xie "Scalable Diffusion Models with Transformers" (2023)
4. Esser et al. "Scaling Rectified Flow Transformers for High-Resolution Image Synthesis" (2024)
5. Lipman et al. "Flow Matching for Generative Modeling" (2022)
6. Liu et al. "Flow Straight and Fast: Learning to Generate and Transfer Data with Rectified Flow" (2022)
7. Black Forest Labs "FLUX.1" (2024)
8. Xiao et al. "FLUX.1 Kontext" (2025)